# Evaluación Parcial N°3 — Transformer encoder-decoder para generación de respuestas

## Generación del turno siguiente en DailyDialog con un Transformer (PyTorch) y comparación con un Seq2Seq LSTM

**Curso:** DLY0100 — Deep Learning<br>
**Sección:** 003D<br>
**Integrantes:** Esteban Pacheco Vega y Francisco Figueroa Donoso<br>
**Fecha:** noviembre de 2026<br>
**Repositorio:** https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY010_003D

## 1. Introducción

### 1.1 Descripción del problema

El dataset corresponde a **DailyDialog** (Li et al., 2017): diálogos cotidianos en inglés escritos por personas, divididos en `train` (11.118 diálogos), `validation` (1.000) y `test` (1.000). Aquí la tarea es de **generación de texto**: dado un turno del diálogo, el modelo debe **generar la respuesta siguiente**.

| Entrada (turno t) | Respuesta real (turno t + 1) |
|---|---|
| «say , jim , how about going for a few beers after dinner ?» | «you know that is tempting but is really not good for our fitness .» |
| «who's ellie ?» | «she's my little niece . …» |

Es un problema de **secuencia a secuencia**, como la traducción automática, con una diferencia importante: en traducción cada frase tiene una o pocas traducciones correctas, mientras que un turno de diálogo admite **muchas respuestas válidas**. Por eso las métricas de coincidencia con la referencia (BLEU y ROUGE) resultan mucho más bajas que en traducción, aunque el modelo genere respuestas razonables.

### 1.2 Objetivo del modelo

- Implementar en **PyTorch** un **Transformer encoder-decoder** con atención de múltiples cabezas, explicando cada componente (atención, feed-forward, encoder, decoder y máscaras).
- Visualizar la máscara causal, la predicción del siguiente token y la atención.
- Ajustar los hiperparámetros de a uno (cabezas, capas, dimensión, feed-forward, Dropout, tasa de aprendizaje y tamaño del lote) con validación, midiendo antes el rango de error entre reentrenamientos.
- Evaluar con **BLEU** y **ROUGE**, más métricas de diversidad, y analizar ejemplos de salida.
- Comparar con un **Seq2Seq LSTM** entrenado con los mismos datos, el mismo vocabulario y las mismas métricas.

### 1.3 Estructura del notebook

1. Introducción.
2. Configuración inicial.
3. Carga de los datos y construcción de pares.
4. Preprocesamiento (vocabulario, lotes y relleno).
5. Componentes del Transformer.
6. Utilidades de entrenamiento, generación, métricas y experimentos.
7. Modelo base.
8. Rango de error entre reentrenamientos.
9. Visualización de la máscara y de la predicción del siguiente token.
10. Experimentos de ajuste.
11. Modelo final y estrategia de decodificación.
12. Evaluación en test.
13. Comparación con un Seq2Seq LSTM.
14. Comparación entre la configuración base y la final.
15. Síntesis de los experimentos.
16. Verificaciones de la dependencia del orden.
17. Conclusiones y mejoras propuestas.
18. Conclusión final.
19. Referencias.
20. Anexo A: síntesis visual para la presentación.
21. Anexo B: herramientas y tecnologías de apoyo utilizadas.
22. Respaldo de resultados.

**Criterio metodológico:** las decisiones se toman con **validación**; el conjunto de **test** se usa una sola vez por modelo final. Todas las corridas usan el mismo tope de épocas y la misma paciencia; solo se extienden si una variante llega al tope todavía mejorando.

## 2. Configuración inicial

Se importan en un único bloque todas las librerías del notebook:

- **PyTorch**: construcción y entrenamiento de los modelos, con control explícito del ciclo de entrenamiento y uso de GPU.
- **NLTK**: cálculo de BLEU, una implementación estándar y probada de la métrica.
- **NumPy / Pandas**: arreglos y tablas comparativas.
- **Plotly**: gráficos interactivos; el renderizador `vscode+colab` permite verlos en Colab y en VS Code conectado a Colab.
- **IPython.display**: tablas con `display`.
- **re / time / random / copy / json / math / os / shutil / urllib / warnings / collections**: lectura del texto, tiempos, semillas, copia de los mejores pesos, resultados guardados y respaldo.

ROUGE y las métricas de diversidad se implementan en el notebook, porque son fórmulas breves y así no se depende de librerías adicionales.

In [ ]:
# ============================================================
# Importación de librerías (todos los imports del notebook van aquí)
# ============================================================
import copy
import json
import math
import os
import random
import re
import shutil
import time
import urllib.request
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
from IPython.display import display
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu, sentence_bleu
from plotly.subplots import make_subplots
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader
from torch.utils.data.sampler import Sampler

warnings.filterwarnings("ignore", category=UserWarning, module="nltk")

# ============================================================
# Reproducibilidad, dispositivo y visualización
# ============================================================
SEED = 42


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pio.renderers.default = "vscode+colab"
pd.set_option("display.max_colwidth", 120)
COLORES = ["#104281", "#3987E5", "#86B6EF"]
AMARILLO = "#F7B500"
print("PyTorch", torch.__version__, "| dispositivo:", device)

**Requisito de ejecución:** activar GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*). Cada entrenamiento del Transformer toma varios minutos.

### 2.1 Reutilización de resultados ya obtenidos

Varias corridas de este notebook se ejecutaron previamente con las mismas configuraciones y semillas. Sus métricas se guardaron en `resultados_transformer_guardados.json` (en el repositorio) y se reutilizan cuando la configuración, la semilla, el tope de épocas y la paciencia coinciden **exactamente**.

- Las corridas que necesitan el modelo entrenado (modelo base, modelo final y Seq2Seq LSTM) se entrenan siempre.
- Cualquier configuración nueva (ampliaciones de rango, extensiones de épocas, semillas adicionales y verificaciones del orden) se entrena.
- Cada fila indica su `origen`: `entrenado` o `guardado`. Algunas corridas guardadas de los experimentos 10.6 y 10.7 conservan solo las métricas que se mostraron en su tabla; las demás aparecen vacías.
- El JSON se reescribe después de cada entrenamiento, con todas las corridas acumuladas. Si Google Drive está montado (botón *Montar Drive* del panel de archivos de Colab), también se guarda en `MyDrive/EP3/`, y al iniciar se lee primero desde ahí: si la sesión se corta, basta con volver a ejecutar y lo ya entrenado se reutiliza.
- Con `REUTILIZAR_RESULTADOS = False` se entrena todo desde cero.

In [ ]:
# ============================================================
# Carga de los resultados guardados
# ============================================================
REUTILIZAR_RESULTADOS = True
ARCHIVO_RESULTADOS = "resultados_transformer_guardados.json"
URL_REPO = "https://raw.githubusercontent.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY010_003D/main/"


CARPETA_DRIVE = "/content/drive/MyDrive/EP3"


def leer_archivo_repo(nombre):
    for ruta in [os.path.join(CARPETA_DRIVE, nombre), nombre]:
        if os.path.exists(ruta):
            print("Leído desde", ruta)
            with open(ruta, encoding="utf-8") as f:
                return f.read()
    print("Leído desde el repositorio")
    with urllib.request.urlopen(URL_REPO + nombre) as r:
        return r.read().decode("utf-8")


RESULTADOS_GUARDADOS = json.loads(leer_archivo_repo(ARCHIVO_RESULTADOS)) if REUTILIZAR_RESULTADOS else {}
CORRIDAS_NUEVAS = {}
print(f"Corridas guardadas disponibles: {len(RESULTADOS_GUARDADOS)}")
print("Google Drive montado:", os.path.isdir("/content/drive/MyDrive"))

## 3. Carga de los datos y construcción de pares

### 3.1 Lectura de los archivos y parser de diálogos

Los archivos se leen desde el repositorio del equipo (o desde una copia local). La columna `dialog` es la representación en texto de un arreglo de NumPy, sin comas entre los turnos, por lo que `ast.literal_eval` no sirve: `parsear_dialogo` extrae cada turno con una expresión regular que respeta las comillas escapadas (`\'`).

In [ ]:
# ============================================================
# Lectura de las particiones y parser
# ============================================================
PATRON_TURNO = re.compile(r"'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\"", re.S)
ACTOS = ["inform", "question", "directive", "commissive"]


def cargar_particion(nombre):
    ruta_local = f"{nombre}.csv"
    return pd.read_csv(ruta_local if os.path.exists(ruta_local) else URL_REPO + ruta_local)


def parsear_dialogo(texto):
    turnos = [(a or b) for a, b in PATRON_TURNO.findall(texto)]
    return [t.replace("\\'", "'").replace('\\"', '"').strip() for t in turnos]


def parsear_etiquetas(texto):
    return [int(x) for x in texto.strip("[]").split()]


df_train = cargar_particion("train")
df_val = cargar_particion("validation")
df_test = cargar_particion("test")
print(f"Diálogos: train {len(df_train):,} | validation {len(df_val):,} | test {len(df_test):,}")

### 3.2 Pares turno → respuesta

`construir_pares(df)` forma un par por cada turno con su respuesta inmediata dentro del mismo diálogo. Cada turno se pasa a minúsculas, se separa por espacios (la puntuación ya viene separada) y se trunca a `MAX_TOKENS_TURNO = 40` tokens, que cubre ≈ 97,5 % de los turnos completos.

Se guarda también el **acto de diálogo del turno de entrada** (*inform*, *question*, *directive* o *commissive*). No se usa para entrenar; sirve para analizar en qué tipo de turnos el modelo responde mejor (sección 12.3).

In [ ]:
# ============================================================
# Construcción de pares turno -> respuesta
# ============================================================
MAX_TOKENS_TURNO = 40


def construir_pares(df):
    pares = []
    for i, fila in df.iterrows():
        turnos = [t.lower().split()[:MAX_TOKENS_TURNO] for t in parsear_dialogo(fila["dialog"])]
        actos = parsear_etiquetas(fila["act"])
        for k in range(len(turnos) - 1):
            if turnos[k] and turnos[k + 1]:
                pares.append({"dialogo_id": i, "entrada": turnos[k], "respuesta": turnos[k + 1],
                              "acto_entrada": ACTOS[actos[k] - 1] if k < len(actos) else None})
    return pd.DataFrame(pares)


pares_train = construir_pares(df_train)
pares_val = construir_pares(df_val)
pares_test = construir_pares(df_test)
display(pd.DataFrame({"particion": ["train", "validation", "test"],
                      "pares": [len(pares_train), len(pares_val), len(pares_test)]}))

In [ ]:
# ============================================================
# Primeros pares de train
# ============================================================
muestra = pares_train.head(5).copy()
muestra["entrada"] = muestra["entrada"].str.join(" ")
muestra["respuesta"] = muestra["respuesta"].str.join(" ")
display(muestra)

**Observación:** *(completar después de ejecutar)*

- Pares por partición (se esperan 76.052, 7.069 y 6.740).

### 3.3 Exploración de las respuestas

Si unas pocas respuestas genéricas dominan `train`, el modelo tenderá a repetirlas, porque minimizan la pérdida promedio sin depender de la entrada.

In [ ]:
# ============================================================
# Respuestas más frecuentes y largo de las respuestas
# ============================================================
largos_respuesta = pares_train["respuesta"].str.len()
print(f"Largo de la respuesta: media {largos_respuesta.mean():.1f} | p50 {largos_respuesta.median():.0f} "
      f"| p95 {largos_respuesta.quantile(0.95):.0f} tokens")
frecuentes = pares_train["respuesta"].str.join(" ").value_counts().head(12).to_frame("veces")
frecuentes["% de las respuestas"] = (frecuentes["veces"] / len(pares_train) * 100).round(2)
display(frecuentes)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Respuestas genéricas más frecuentes y su peso; riesgo de que el modelo las repita.

## 4. Preprocesamiento

### 4.1 Vocabulario compartido

`Vocab` asigna un índice a cada token que aparece al menos `MIN_FREQ` veces y envía el resto a `<unk>`. Es una clase mínima propia en vez de `torchtext`, librería descontinuada e incompatible con las versiones actuales de PyTorch.

- Entrada y respuesta están en el mismo idioma, así que se usa **un solo vocabulario** para el encoder y el decoder: las palabras comparten su representación y el vocabulario no se duplica.
- Se construye solo con los turnos de `train`, contando cada turno una vez.
- `MIN_FREQ = 3`: se conservan los tokens que aparecen al menos 3 veces (≈ 9.900 tokens); los más raros casi no se entrenarían.
- Tokens especiales: `<unk>` (desconocido), `<eos>` (fin), `<bos>` (inicio) y `<pad>` (relleno). La entrada termina en `<eos>`; la respuesta empieza con `<bos>` y termina con `<eos>`.

In [ ]:
# ============================================================
# Vocabulario compartido
# ============================================================
class Vocab:
    def __init__(self, counter, min_freq=1, specials=("<unk>",)):
        self.itos = list(specials) + sorted(t for t, c in counter.items() if c >= min_freq and t not in specials)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.default_index = self.stoi["<unk>"]

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __len__(self):
        return len(self.itos)

    def get_itos(self):
        return self.itos


MIN_FREQ = 3
ESPECIALES = ("<unk>", "<eos>", "<bos>", "<pad>")
contador = Counter(tok for d in df_train["dialog"] for t in parsear_dialogo(d)
                   for tok in t.lower().split()[:MAX_TOKENS_TURNO])
vocab = Vocab(contador, min_freq=MIN_FREQ, specials=ESPECIALES)
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = vocab["<pad>"], vocab["<bos>"], vocab["<eos>"], vocab["<unk>"]
print(f"Tokens distintos en train: {len(contador):,} | vocabulario (min_freq = {MIN_FREQ}): {len(vocab):,}")

### 4.2 Formato de los datos

Cada par se convierte en una tupla `(origen, largo_origen, destino)` con los tokens especiales ya agregados.

In [ ]:
# ============================================================
# Formato (origen, largo, destino)
# ============================================================
def a_formato(pares):
    return [(e + ["<eos>"], len(e) + 1, ["<bos>"] + r + ["<eos>"]) for e, r in zip(pares["entrada"], pares["respuesta"])]


train_list = a_formato(pares_train)
val_list = a_formato(pares_val)
test_list = a_formato(pares_test)
src, largo, trg = train_list[0]
print("src =", src)
print("trg =", trg)
tokens_val = [t for s, _, _ in val_list for t in s]
print(f"Tokens de validación fuera del vocabulario: {np.mean([vocab[t] == UNK_IDX for t in tokens_val]):.2%}")

### 4.3 Lotes por largo y relleno

- `BucketSampler` arma lotes con frases de **largo parecido**, para que el relleno sea mínimo y cada lote se procese más rápido. En entrenamiento, el orden de los lotes se baraja en cada época.
- `collate_batch` convierte los tokens en índices y rellena con `<pad>`, con tensores de forma **[batch size, largo]**.

In [ ]:
# ============================================================
# BucketSampler, collate_batch y DataLoaders
# ============================================================
class BucketSampler(Sampler):
    def __init__(self, batch_size, datos, semilla=SEED):
        self.rng = random.Random(semilla)
        indices = [(i, d[1]) for i, d in enumerate(datos)]
        self.rng.shuffle(indices)
        agrupados = []
        for i in range(0, len(indices), batch_size * 100):
            agrupados.extend(sorted(indices[i:i + batch_size * 100], key=lambda x: x[1], reverse=True))
        self.lotes = [[idx for idx, _ in agrupados[i:i + batch_size]] for i in range(0, len(agrupados), batch_size)]

    def __iter__(self):
        self.rng.shuffle(self.lotes)
        return iter(self.lotes)

    def __len__(self):
        return len(self.lotes)


def collate_batch(batch):
    text_src = [torch.tensor([vocab[t] for t in src]) for src, _, _ in batch]
    text_tgt = [torch.tensor([vocab[t] for t in tgt]) for _, _, tgt in batch]
    return (pad_sequence(text_src, batch_first=True, padding_value=PAD_IDX),
            pad_sequence(text_tgt, batch_first=True, padding_value=PAD_IDX))


def crear_loader(datos, batch_size, semilla=SEED):
    return DataLoader(datos, batch_sampler=BucketSampler(batch_size, datos, semilla), collate_fn=collate_batch)


src_lote, trg_lote = next(iter(crear_loader(train_list, 64)))
print("src:", tuple(src_lote.shape), "| trg:", tuple(trg_lote.shape))

## 5. Componentes del Transformer

La implementación sigue la arquitectura de *Attention Is All You Need* (Vaswani et al., 2017), con tres diferencias: la codificación posicional es **aprendida** (como en BERT y GPT-2), la tasa de aprendizaje es fija (sin calentamiento) y no se usa suavizado de etiquetas. Las clases conservan nombres estándar de implementaciones de referencia.

| Clase | Parte de la arquitectura |
|---|---|
| `MultiHeadAttentionLayer` | Q, K y V · división en cabezas · atención · concatenación |
| `PositionwiseFeedforwardLayer` | Feed-forward de cada bloque |
| `EncoderBlock`, `Encoder` | Codificador |
| `DecoderLayer`, `Decoder` | Decodificador |
| `TransformerSeq2Seq` | Une ambos lados y crea las máscaras |

### 5.1 Atención de múltiples cabezas

La atención compara cada **consulta** (*query*, Q) con todas las **claves** (*keys*, K) y usa esos puntajes para mezclar los **valores** (*values*, V):

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

- Q, K y V se obtienen con tres capas lineales (`fc_q`, `fc_k`, `fc_v`).
- La dimensión `hid_dim` se divide en `n_heads` **cabezas** de tamaño `head_dim = hid_dim / n_heads`. Cada cabeza calcula su propia atención en paralelo y puede aprender relaciones distintas.
- Los puntajes se dividen por $\sqrt{d_k}$ para que el softmax no se sature.
- La **máscara** reemplaza por −10¹⁰ las posiciones que no se deben mirar (relleno o futuro), de modo que su peso tras el softmax sea 0.
- Las salidas de las cabezas se concatenan y se combinan con `fc_o` ($W^O$).

En el Transformer hay **tres usos** de esta misma capa: autoatención en el encoder, autoatención **con máscara** en el decoder y atención **cruzada** decoder → encoder. Cambia solo de dónde salen Q, K y V.

In [ ]:
# ============================================================
# Atención de múltiples cabezas
# ============================================================
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout, device):
        super().__init__()
        assert hid_dim % n_heads == 0
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads
        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)
        self.fc_o = nn.Linear(hid_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim])).to(device)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)
        # Q = [batch size, query len, hid dim]
        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # Q = [batch size, n heads, query len, head dim]
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        # energy = [batch size, n heads, query len, key len]
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attention), V)
        # x = [batch size, n heads, query len, head dim]
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.hid_dim)
        x = self.fc_o(x)
        # x = [batch size, query len, hid dim]
        return x, attention

### 5.2 Feed-forward posicional

Después de la atención, cada token pasa **por separado** por un MLP de dos capas: `hid_dim → pf_dim → hid_dim`, con ReLU y Dropout. La atención **mezcla** información entre tokens; el feed-forward **procesa** lo que cada token reunió. El paper original usa 512 → 2.048.

In [ ]:
# ============================================================
# Feed-forward posicional
# ============================================================
class PositionwiseFeedforwardLayer(nn.Module):
    def __init__(self, hid_dim, pf_dim, dropout):
        super().__init__()
        self.fc_1 = nn.Linear(hid_dim, pf_dim)
        self.fc_2 = nn.Linear(pf_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(torch.relu(self.fc_1(x)))
        # x = [batch size, seq len, pf dim]
        return self.fc_2(x)

### 5.3 Encoder

**Bloque encoder:** autoatención sobre la frase de entrada → Dropout + **conexión residual** + **LayerNorm** → feed-forward → Dropout + residual + LayerNorm. La conexión residual suma la entrada de cada subcapa a su salida (facilita entrenar redes profundas) y LayerNorm normaliza cada vector de token (estabiliza el entrenamiento).

**Encoder:** a diferencia de una RNN, no comprime la frase en un solo vector de contexto: produce **un vector por token**, y cada uno ya «vio» todos los tokens de la frase.

- **Embedding del token** (escalado por $\sqrt{d_{modelo}}$) + **embedding de posición aprendido**. Sin la posición, el modelo no sabría el orden de las palabras, porque no tiene recurrencia.
- `max_length = 100`: el modelo acepta frases de hasta 100 tokens (aquí, como máximo 43).
- La máscara de origen (`src_mask`) evita prestar atención a `<pad>`.

In [ ]:
# ============================================================
# Bloque encoder y encoder
# ============================================================
class EncoderBlock(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout, device):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        # src = [batch size, src len, hid dim]; src_mask = [batch size, 1, 1, src len]
        _src, _ = self.self_attention(src, src, src, src_mask)
        src = self.self_attn_layer_norm(src + self.dropout(_src))
        _src = self.positionwise_feedforward(src)
        src = self.ff_layer_norm(src + self.dropout(_src))
        return src


class Encoder(nn.Module):
    def __init__(self, input_dim, hid_dim, n_layers, n_heads, pf_dim, dropout, device, max_length=100):
        super().__init__()
        self.device = device
        self.tok_embedding = nn.Embedding(input_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)
        self.layers = nn.ModuleList([EncoderBlock(hid_dim, n_heads, pf_dim, dropout, device) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

    def forward(self, src, src_mask):
        batch_size, src_len = src.shape
        pos = torch.arange(0, src_len).unsqueeze(0).repeat(batch_size, 1).to(self.device)
        src = self.dropout((self.tok_embedding(src) * self.scale) + self.pos_embedding(pos))
        for layer in self.layers:
            src = layer(src, src_mask)
        # src = [batch size, src len, hid dim]
        return src

### 5.4 Decoder

**Bloque decoder:** tiene **dos** capas de atención:

1. **Autoatención con máscara** sobre la respuesta generada hasta el momento: cada token solo puede mirar los anteriores (máscara causal), para que al entrenar no «copie» el token que debe predecir.
2. **Atención cruzada** (`encoder_attention`): las consultas salen del decoder y las claves y valores, de la salida del encoder. Al generar cada palabra, el decoder decide en qué palabras de la entrada fijarse.

Luego viene el feed-forward, con Dropout, residual y LayerNorm después de cada subcapa.

**Decoder:** embeddings de token y posición, `n_layers` bloques y una capa lineal final (`fc_out`) que produce un **logit por cada token del vocabulario**. El softmax está dentro de `CrossEntropyLoss`. Devuelve también la atención cruzada de la última capa, para visualizarla.

In [ ]:
# ============================================================
# Bloque decoder y decoder
# ============================================================
class DecoderLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, pf_dim, dropout, device):
        super().__init__()
        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.enc_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.encoder_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        _trg, _ = self.self_attention(trg, trg, trg, trg_mask)
        trg = self.self_attn_layer_norm(trg + self.dropout(_trg))
        _trg, attention = self.encoder_attention(trg, enc_src, enc_src, src_mask)
        trg = self.enc_attn_layer_norm(trg + self.dropout(_trg))
        _trg = self.positionwise_feedforward(trg)
        trg = self.ff_layer_norm(trg + self.dropout(_trg))
        # attention = [batch size, n heads, trg len, src len]
        return trg, attention


class Decoder(nn.Module):
    def __init__(self, output_dim, hid_dim, n_layers, n_heads, pf_dim, dropout, device, max_length=100):
        super().__init__()
        self.device = device
        self.tok_embedding = nn.Embedding(output_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)
        self.layers = nn.ModuleList([DecoderLayer(hid_dim, n_heads, pf_dim, dropout, device) for _ in range(n_layers)])
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        batch_size, trg_len = trg.shape
        pos = torch.arange(0, trg_len).unsqueeze(0).repeat(batch_size, 1).to(self.device)
        trg = self.dropout((self.tok_embedding(trg) * self.scale) + self.pos_embedding(pos))
        for layer in self.layers:
            trg, attention = layer(trg, enc_src, trg_mask, src_mask)
        output = self.fc_out(trg)
        # output = [batch size, trg len, output dim]
        return output, attention

### 5.5 Modelo completo y máscaras

`TransformerSeq2Seq` une encoder y decoder y crea las dos máscaras:

- **`make_src_mask`**: 1 donde la entrada no es `<pad>`.
- **`make_trg_mask`**: combina (AND lógico) la máscara de relleno con una matriz **triangular inferior** (`torch.tril`): la fila *i* solo puede mirar las columnas 0 a *i*.

`construir_transformer(config)` arma el modelo con los hiperparámetros de la configuración e inicializa los pesos con **Xavier**, que mantiene estable la varianza de las activaciones entre capas al inicio del entrenamiento.

In [ ]:
# ============================================================
# Modelo completo y construcción
# ============================================================
class TransformerSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, trg_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self, src):
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)
        trg_len = trg.shape[1]
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=self.device)).bool()
        return trg_pad_mask & trg_sub_mask

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_src = self.encoder(src, src_mask)
        output, attention = self.decoder(trg, enc_src, trg_mask, src_mask)
        return output, attention


def initialize_weights(m):
    if hasattr(m, "weight") and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)


def construir_transformer(config):
    enc = Encoder(len(vocab), config["hid_dim"], config["capas"], config["cabezas"], config["pf_dim"], config["dropout"], device)
    dec = Decoder(len(vocab), config["hid_dim"], config["capas"], config["cabezas"], config["pf_dim"], config["dropout"], device)
    modelo = TransformerSeq2Seq(enc, dec, PAD_IDX, PAD_IDX, device).to(device)
    modelo.apply(initialize_weights)
    return modelo


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)

## 6. Utilidades de entrenamiento, generación, métricas y experimentos

### 6.1 Entrenamiento y evaluación por época

- **Teacher forcing:** el decoder recibe la respuesta real sin el último token (`trg[:, :-1]`) y debe predecir la respuesta sin `<bos>` (`trg[:, 1:]`). Gracias a la máscara causal, todas las posiciones se predicen **en paralelo**.
- La pérdida es `CrossEntropyLoss(ignore_index=PAD_IDX)`: el relleno no cuenta.
- Se recorta el gradiente (`clip_grad_norm_`), para evitar actualizaciones bruscas, y se usa Adam.
- `evaluar` devuelve además la **exactitud por token** (sin contar el relleno).
- La **perplejidad** (PPL) es exp(pérdida): el número de tokens entre los que el modelo «duda» en promedio. Menor es mejor.

In [ ]:
# ============================================================
# Entrenamiento y evaluación de una época
# ============================================================
def entrenar_epoca(modelo, iterador, optimizador, criterio, clip):
    modelo.train()
    epoch_loss = 0
    for src, trg in iterador:
        src, trg = src.to(device), trg.to(device)
        optimizador.zero_grad()
        output, _ = modelo(src, trg[:, :-1])
        loss = criterio(output.reshape(-1, output.shape[-1]), trg[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), clip)
        optimizador.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterador)


def evaluar(modelo, iterador, criterio):
    modelo.eval()
    epoch_loss, aciertos, tokens = 0, 0, 0
    with torch.no_grad():
        for src, trg in iterador:
            src, trg = src.to(device), trg.to(device)
            output, _ = modelo(src, trg[:, :-1])
            objetivo = trg[:, 1:]
            epoch_loss += criterio(output.reshape(-1, output.shape[-1]), objetivo.reshape(-1)).item()
            mascara = objetivo != PAD_IDX
            aciertos += ((output.argmax(-1) == objetivo) & mascara).sum().item()
            tokens += mascara.sum().item()
    return epoch_loss / len(iterador), aciertos / tokens

### 6.2 Generación de respuestas

`generar_lote(modelo, frases_origen, temperatura, top_k)` genera las respuestas **token a token**:

1. Empieza cada respuesta con `<bos>`.
2. Pasa la entrada y la respuesta parcial por el modelo y toma los **logits** del último token.
3. Elige el siguiente token: el más probable (*greedy*, `temperatura=None`) o un **sorteo** entre los `top_k` más probables, con los logits divididos por la temperatura (temperatura baja → más predecible; alta → más variado).
4. Se detiene al generar `<eos>` o al llegar a 42 tokens.

Funciona con cualquier modelo que reciba `(src, trg)` y devuelva `(logits, atención)`, por lo que sirve también para el Seq2Seq LSTM de la sección 13. `generar_con_atencion` genera una sola respuesta y devuelve además la atención cruzada de la última capa.

In [ ]:
# ============================================================
# Generación token a token (greedy o sorteo con temperatura y top-k)
# ============================================================
MAX_LARGO_GENERACION = MAX_TOKENS_TURNO + 2


def generar_lote(modelo, src_lists, temperatura=None, top_k=None, max_len=MAX_LARGO_GENERACION):
    modelo.eval()
    seqs = [torch.tensor([vocab[t] for t in s]) for s in src_lists]
    src = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX).to(device)
    trg = torch.full((len(src_lists), 1), BOS_IDX, dtype=torch.long, device=device)
    terminado = torch.zeros(len(src_lists), dtype=torch.bool, device=device)
    with torch.no_grad():
        for _ in range(max_len):
            output, _ = modelo(src, trg)
            logits = output[:, -1]
            if temperatura is None:
                siguiente = logits.argmax(-1)
            else:
                logits = logits / temperatura
                if top_k is not None:
                    umbral = torch.topk(logits, top_k, dim=-1).values[:, -1:]
                    logits = logits.masked_fill(logits < umbral, float("-inf"))
                siguiente = torch.multinomial(torch.softmax(logits, dim=-1), 1).squeeze(1)
            siguiente = siguiente.masked_fill(terminado, EOS_IDX)
            trg = torch.cat([trg, siguiente.unsqueeze(1)], dim=1)
            terminado |= siguiente == EOS_IDX
            if terminado.all():
                break
    itos = vocab.get_itos()
    respuestas = []
    for fila in trg[:, 1:].tolist():
        tokens = []
        for i in fila:
            if i == EOS_IDX:
                break
            tokens.append(itos[i])
        respuestas.append(tokens)
    return respuestas


def generar_respuestas(modelo, datos, temperatura=None, top_k=None, batch_size=256):
    respuestas = []
    for i in range(0, len(datos), batch_size):
        lote = datos[i:i + batch_size]
        respuestas += generar_lote(modelo, [src for src, _, _ in lote], temperatura, top_k)
    return respuestas


def generar_con_atencion(modelo, tokens_origen, max_len=MAX_LARGO_GENERACION):
    modelo.eval()
    src = torch.tensor([[vocab[t] for t in tokens_origen]], device=device)
    src_mask = modelo.make_src_mask(src)
    with torch.no_grad():
        enc_src = modelo.encoder(src, src_mask)
    trg_indexes = [BOS_IDX]
    for _ in range(max_len):
        trg = torch.tensor([trg_indexes], device=device)
        with torch.no_grad():
            output, attention = modelo.decoder(trg, enc_src, modelo.make_trg_mask(trg), src_mask)
        trg_indexes.append(output[0, -1].argmax().item())
        if trg_indexes[-1] == EOS_IDX:
            break
    itos = vocab.get_itos()
    return [itos[i] for i in trg_indexes[1:]], attention

### 6.3 Métricas de generación

Las métricas comparan cada respuesta generada con **la** respuesta real del diálogo:

- **BLEU** (*Bilingual Evaluation Understudy*, `corpus_bleu` de NLTK): precisión de n-gramas de 1 a 4 palabras de la respuesta generada respecto de la real, con penalización si la respuesta es más corta que la referencia. Se reporta también **BLEU-1** (solo palabras sueltas). Se usa suavizado (`SmoothingFunction().method1`) para que la ausencia de 4-gramas en común no deje el puntaje en 0.
- **ROUGE-1 y ROUGE-2** (F): solapamiento de palabras y de pares de palabras, combinando precisión y exhaustividad.
- **ROUGE-L** (F): basado en la **subsecuencia común más larga**; premia el orden de las palabras sin exigir que sean contiguas.
- **distinct-1 y distinct-2:** proporción de palabras y pares de palabras **distintos** entre todas las respuestas generadas. Detectan si el modelo repite siempre las mismas respuestas genéricas. **Dependen del número de respuestas** (con más respuestas, más repeticiones), así que solo se comparan con la misma cantidad.
- **Largo medio** de las respuestas generadas.

Como un turno admite muchas respuestas válidas y hay **una sola** referencia, BLEU y ROUGE subestiman la calidad: sirven para **comparar modelos con los mismos datos**, no como nota absoluta. En generación de diálogo es habitual un BLEU de un dígito.

Para que los experimentos sean rápidos, estas métricas se calculan sobre una **muestra fija de 1.000 pares de validación**, la misma en todas las variantes.

In [ ]:
# ============================================================
# BLEU (NLTK), ROUGE y diversidad
# ============================================================
SUAVIZADO = SmoothingFunction().method1


def ngramas(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def f_desde(solapamiento, n_hipotesis, n_referencia):
    if solapamiento == 0:
        return 0.0
    precision, exhaustividad = solapamiento / n_hipotesis, solapamiento / n_referencia
    return 2 * precision * exhaustividad / (precision + exhaustividad)


def rouge_n(referencia, hipotesis, n):
    r, h = ngramas(referencia, n), ngramas(hipotesis, n)
    return f_desde(sum((r & h).values()), sum(h.values()), sum(r.values()))


def rouge_l(referencia, hipotesis):
    if not referencia or not hipotesis:
        return 0.0
    tabla = [[0] * (len(hipotesis) + 1) for _ in range(len(referencia) + 1)]
    for i, a in enumerate(referencia, 1):
        for j, b in enumerate(hipotesis, 1):
            tabla[i][j] = tabla[i - 1][j - 1] + 1 if a == b else max(tabla[i - 1][j], tabla[i][j - 1])
    return f_desde(tabla[-1][-1], len(hipotesis), len(referencia))


def distinct_n(hipotesis, n):
    todos = [g for h in hipotesis for g in ngramas(h, n).elements()]
    return len(set(todos)) / len(todos) if todos else 0.0


def evaluar_generacion(referencias, hipotesis):
    refs = [[r] for r in referencias]
    return {
        "bleu": corpus_bleu(refs, hipotesis, smoothing_function=SUAVIZADO),
        "bleu_1": corpus_bleu(refs, hipotesis, weights=(1, 0, 0, 0), smoothing_function=SUAVIZADO),
        "rouge_1": np.mean([rouge_n(r, h, 1) for r, h in zip(referencias, hipotesis)]),
        "rouge_2": np.mean([rouge_n(r, h, 2) for r, h in zip(referencias, hipotesis)]),
        "rouge_l": np.mean([rouge_l(r, h) for r, h in zip(referencias, hipotesis)]),
        "distinct_1": distinct_n(hipotesis, 1),
        "distinct_2": distinct_n(hipotesis, 2),
        "largo_medio": np.mean([len(h) for h in hipotesis]),
    }


MUESTRA_VAL = random.Random(SEED).sample(val_list, 1000)
REFERENCIAS_MUESTRA = [trg[1:-1] for _, _, trg in MUESTRA_VAL]
display(pd.DataFrame([evaluar_generacion([["i", "am", "fine", ",", "thanks", "."]], [["i", "am", "fine", "."]])]).round(4))

La tabla anterior es una prueba de las métricas con una respuesta inventada: «i am fine .» frente a la referencia «i am fine , thanks .».

### 6.4 Entrenamiento con Early Stopping y registro de métricas

`entrenar_y_registrar(config, constructor, semilla, epocas, paciencia, verbose, forzar)` entrena con **Early Stopping**: se detiene si la pérdida de validación no mejora durante `paciencia` épocas y, al terminar, restaura los mejores pesos y calcula las métricas de generación sobre la muestra de validación.

- **Régimen uniforme:** todas las corridas, incluido el modelo final y el Seq2Seq LSTM, usan `EPOCAS_EXPERIMENTO = 15` y `PACIENCIA_EXPERIMENTO = 2`.
- **Extensión:** si la mejor época queda en el tope (todavía mejorando), la corrida se repite con `EPOCAS_EXTENDIDAS = 25`, y esa es la que se reporta (`epocas_tope = 25`). Así ninguna variante pierde por falta de épocas.
- Si existe un resultado guardado con la misma clave y `forzar=False`, se reutiliza (sin modelo ni historia). Cada corrida nueva se agrega al JSON de resultados de la sesión.
- **`constructor`**: `construir_transformer` o, en la sección 13, `construir_lstm`.
- **Retorna** `(fila, historia, modelo)`. `fila` contiene: pérdida de entrenamiento y validación, perplejidad, exactitud por token, BLEU, ROUGE, distinct, largo medio, mejor época, épocas entrenadas, tope de épocas, parámetros, tiempos y origen.

In [ ]:
# ============================================================
# Entrenamiento con Early Stopping, extensión y registro de métricas
# ============================================================
EPOCAS_EXPERIMENTO = 15
PACIENCIA_EXPERIMENTO = 2
EPOCAS_EXTENDIDAS = 25


def clave_corrida(config, semilla, epocas, paciencia):
    return json.dumps({**config, "semilla": semilla, "epocas": epocas, "paciencia": paciencia}, sort_keys=True)


def guardar_corridas():
    destinos = [ARCHIVO_RESULTADOS]
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(CARPETA_DRIVE, exist_ok=True)
        destinos.append(os.path.join(CARPETA_DRIVE, ARCHIVO_RESULTADOS))
    for destino in destinos:
        with open(destino, "w", encoding="utf-8") as f:
            json.dump({**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}, f, ensure_ascii=False, indent=1,
                      default=lambda o: o.item() if hasattr(o, "item") else str(o))


def entrenar_y_registrar(config, constructor=None, semilla=SEED, epocas=None, paciencia=None, verbose=False,
                         forzar=False, extender=True):
    constructor = constructor or construir_transformer
    epocas = epocas or EPOCAS_EXPERIMENTO
    paciencia = PACIENCIA_EXPERIMENTO if paciencia is None else paciencia
    clave = clave_corrida(config, semilla, epocas, paciencia)

    if not forzar and clave in RESULTADOS_GUARDADOS:
        fila, historia, modelo = {**RESULTADOS_GUARDADOS[clave], "epocas_tope": epocas, "origen": "guardado"}, None, None
    else:
        fijar_semilla(semilla)
        train_iter = crear_loader(train_list, config["batch_size"], semilla)
        val_iter = crear_loader(val_list, config["batch_size"], semilla)
        modelo = constructor(config)
        optimizador = torch.optim.Adam(modelo.parameters(), lr=config["learning_rate"])
        criterio = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
        historia = {"loss_train": [], "loss_val": [], "exactitud_token_val": []}
        mejor_loss, mejor_estado, mejor_epoca, sin_mejora = float("inf"), None, 0, 0
        inicio = time.time()
        for epoca in range(epocas):
            t0 = time.time()
            train_loss = entrenar_epoca(modelo, train_iter, optimizador, criterio, config["clip"])
            valid_loss, valid_acc = evaluar(modelo, val_iter, criterio)
            for k, valor in zip(historia, [train_loss, valid_loss, valid_acc]):
                historia[k].append(valor)
            if verbose:
                print(f"Época {epoca + 1:02} | {time.time() - t0:.0f} s | train loss {train_loss:.3f} (PPL {math.exp(train_loss):.1f}) "
                      f"| val loss {valid_loss:.3f} (PPL {math.exp(valid_loss):.1f}) | exactitud por token {valid_acc:.3f}")
            if valid_loss < mejor_loss:
                mejor_loss, mejor_epoca, sin_mejora = valid_loss, epoca, 0
                mejor_estado = copy.deepcopy(modelo.state_dict())
            else:
                sin_mejora += 1
                if paciencia and sin_mejora >= paciencia:
                    break
        duracion = time.time() - inicio
        modelo.load_state_dict(mejor_estado)
        generadas = generar_respuestas(modelo, MUESTRA_VAL)
        m = mejor_epoca
        fila = {
            "loss_train": historia["loss_train"][m], "loss_val": historia["loss_val"][m],
            "ppl_val": math.exp(historia["loss_val"][m]), "exactitud_token_val": historia["exactitud_token_val"][m],
            **evaluar_generacion(REFERENCIAS_MUESTRA, generadas),
            "mejor_epoca": m + 1, "epocas_entrenadas": len(historia["loss_train"]),
            "parametros": contar_parametros(modelo), "segundos": duracion,
            "segundos_por_epoca": duracion / len(historia["loss_train"]),
        }
        CORRIDAS_NUEVAS[clave] = dict(fila)
        guardar_corridas()
        fila = {**fila, "epocas_tope": epocas, "origen": "entrenado"}

    if extender and paciencia and fila["mejor_epoca"] >= epocas - 1 and epocas < EPOCAS_EXTENDIDAS:
        print(f"  La mejor época ({fila['mejor_epoca']}) está en el tope de {epocas}: se extiende a {EPOCAS_EXTENDIDAS} épocas.")
        return entrenar_y_registrar(config, constructor, semilla, EPOCAS_EXTENDIDAS, paciencia, verbose, forzar, extender=False)
    return fila, historia, modelo

### 6.5 Experimentos controlados, selección y ampliación del rango

- `ejecutar_experimento(nombre, variantes, config_actual, forzar)` entrena (o recupera) las variantes `{etiqueta: cambios}` y devuelve una tabla ordenada por **menor pérdida de validación** (la métrica que optimiza el entrenamiento y la más estable), con `diferencia_vs_mejor` y `dentro_del_margen` según el rango de error de la sección 8. BLEU y ROUGE-L se usan como confirmación.
- `ampliar_rango(...)`: si la mejor variante queda en el **borde** del rango probado, agrega el valor siguiente en esa dirección y lo entrena; repite mientras el ganador siga en el borde y no se supere el límite indicado.
- `seleccionar_variante(tabla)` propone la variante con menor pérdida y lista las que están en **empate técnico**. Los empates se deciden caso a caso según el contexto; la decisión queda en la línea `# Cambio Manual` y se justifica en la celda de resultado.
- `aplicar_variante(...)` incorpora la elección a `CONFIG_ACTUAL` y la registra en `decisiones`.
- Si se vuelve a ejecutar un experimento, la configuración vuelve a su estado previo y se descartan las decisiones y tablas posteriores, que deben volver a ejecutarse desde ahí hacia abajo.

In [ ]:
# ============================================================
# Experimentos controlados (un hiperparámetro a la vez)
# ============================================================
resultados_experimentos = []
historial_experimentos = []
decisiones = {}
modelos_ultimo_experimento = {}
COLUMNAS_TABLA = ["variante", "loss_val", "ppl_val", "exactitud_token_val", "bleu", "rouge_l", "distinct_2",
                  "mejor_epoca", "epocas_tope", "parametros", "segundos_por_epoca", "diferencia_vs_mejor",
                  "dentro_del_margen", "origen"]


def entrenar_variantes(nombre, variantes, config_actual, forzar=False):
    filas = []
    for etiqueta, cambios in variantes.items():
        config = {**config_actual, **cambios}
        fila, _, modelo = entrenar_y_registrar(config, forzar=forzar)
        print(f"[{nombre}] {etiqueta}: {fila['origen']}")
        if modelo is not None:
            modelos_ultimo_experimento[etiqueta] = (config, modelo)
        filas.append({"experimento": nombre, "variante": etiqueta, **fila})
    return filas


def armar_tabla(filas):
    tabla = pd.DataFrame(filas)
    tabla["diferencia_vs_mejor"] = tabla["loss_val"] - tabla["loss_val"].min()
    tabla["dentro_del_margen"] = tabla["diferencia_vs_mejor"] <= MARGEN_ERROR
    return tabla.sort_values("loss_val").reset_index(drop=True)


def ejecutar_experimento(nombre, variantes, config_actual, forzar=False):
    nombres = [n for n, _ in historial_experimentos]
    if nombre in nombres:
        i = nombres.index(nombre)
        config_actual.clear()
        config_actual.update(historial_experimentos[i][1])
        for n in nombres[i:]:
            decisiones.pop(n, None)
        del historial_experimentos[i:]
        del resultados_experimentos[i:]
        print(f"[{nombre}] Experimento repetido: la configuración vuelve a su estado previo "
              f"y se descartan las decisiones posteriores ({', '.join(nombres[i:])}).")
    historial_experimentos.append((nombre, dict(config_actual)))
    modelos_ultimo_experimento.clear()
    tabla = armar_tabla(entrenar_variantes(nombre, variantes, config_actual, forzar))
    resultados_experimentos.append(tabla)
    return tabla


def ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo):
    tabla = resultados_experimentos[-1]
    while True:
        valores = {etq: cambios[hiperparametro] for etq, cambios in variantes.items()}
        mejor = valores[tabla.loc[0, "variante"]]
        if mejor == max(valores.values()) and arriba(mejor) <= maximo:
            nuevo = arriba(mejor)
        elif mejor == min(valores.values()) and abajo(mejor) >= minimo:
            nuevo = abajo(mejor)
        else:
            return tabla
        print(f"[{nombre}] La mejor variante está en el borde del rango: se agrega {hiperparametro} = {nuevo}")
        variantes[str(nuevo)] = {hiperparametro: nuevo}
        filas = tabla.drop(columns=["diferencia_vs_mejor", "dentro_del_margen"]).to_dict("records")
        filas += entrenar_variantes(nombre, {str(nuevo): {hiperparametro: nuevo}}, config_actual)
        tabla = armar_tabla(filas)
        resultados_experimentos[-1] = tabla


def seleccionar_variante(tabla):
    mejor = tabla.loc[0, "variante"]
    empatadas = tabla.loc[tabla["dentro_del_margen"] & (tabla["variante"] != mejor), "variante"].tolist()
    print(f"Menor pérdida de validación: {mejor}")
    if empatadas:
        print(f"En empate técnico (dentro de {MARGEN_ERROR:.4f} de pérdida): {', '.join(map(str, empatadas))}")
    return mejor


def aplicar_variante(nombre, variantes, elegida):
    CONFIG_ACTUAL.update(variantes[elegida])
    decisiones[nombre] = elegida
    mostrar_config(CONFIG_ACTUAL)

### 6.6 Visualización

- `mostrar_config(config)`: configuración como tabla vertical.
- `graficar_curvas(historia, titulo)`: pérdida y perplejidad por época, con la mejor época marcada.
- `mostrar_atencion(...)`: atención cruzada de cada cabeza. Los ejes usan posiciones numéricas con los tokens como etiquetas, para que dos tokens iguales (por ejemplo, dos «.») no se fusionen en una sola columna.

In [ ]:
# ============================================================
# Configuración, curvas y atención
# ============================================================
def mostrar_config(config):
    display(pd.DataFrame({"valor": pd.Series(config).astype(str)}))


def graficar_curvas(historia, titulo):
    epocas = list(range(1, len(historia["loss_train"]) + 1))
    mejor = int(np.argmin(historia["loss_val"])) + 1
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Pérdida", "Perplejidad"])
    for col, transformar in [(1, np.array), (2, np.exp)]:
        fig.add_trace(go.Scatter(x=epocas, y=transformar(historia["loss_train"]), name="entrenamiento",
                                 line_color=COLORES[2], legendgroup="train", showlegend=col == 1), row=1, col=col)
        fig.add_trace(go.Scatter(x=epocas, y=transformar(historia["loss_val"]), name="validación",
                                 line_color=COLORES[0], legendgroup="val", showlegend=col == 1), row=1, col=col)
        fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
        fig.update_xaxes(title_text="época", row=1, col=col)
    fig.update_layout(title=f"{titulo} · mejor época: {mejor}", height=420)
    fig.show()


def mostrar_atencion(sentence, translation, attention, titulo, cabezas=None, n_cols=2):
    cabezas = cabezas or list(range(min(4, attention.shape[1])))
    n_rows = math.ceil(len(cabezas) / n_cols)
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f"Cabeza {i + 1}" for i in cabezas],
                        horizontal_spacing=0.12, vertical_spacing=0.18)
    for k, i in enumerate(cabezas):
        fila, col = k // n_cols + 1, k % n_cols + 1
        fig.add_trace(go.Heatmap(z=attention[0, i].cpu().numpy(), colorscale="Blues", showscale=False,
                                 hovertemplate="generado %{y}<br>entrada %{x}<br>peso %{z:.3f}<extra></extra>"),
                      row=fila, col=col)
        fig.update_xaxes(tickvals=list(range(len(sentence))), ticktext=sentence, tickangle=45, row=fila, col=col)
        fig.update_yaxes(tickvals=list(range(len(translation))), ticktext=translation, autorange="reversed",
                         row=fila, col=col)
    fig.update_layout(title=titulo, height=380 * n_rows)
    return fig

## 7. Modelo base

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `hid_dim` | 256 | Versión reducida del paper (512), adecuada para ≈ 76.000 pares y una GPU T4 |
| `capas` | 3 | Profundidad intermedia; se prueba de 1 a 3 (y más si hace falta) en la sección 10 |
| `cabezas` | 8 | Mismo número de cabezas que el paper, con `head_dim` = 32 |
| `pf_dim` | 512 | El doble de `hid_dim`, una proporción menor que la del paper (4×) para limitar parámetros |
| `dropout` | 0,1 | Valor del paper |
| `learning_rate` | 0,0005 | Menor que el valor por defecto de Adam (0,001): los Transformers sin calentamiento de la tasa se vuelven inestables con tasas altas |
| `batch_size` | 64 | ≈ 1.189 actualizaciones por época con 76.052 pares |
| `clip` | 1 | Limita la norma del gradiente |

El modelo base se entrena siempre, porque sus pesos se usan en la sección 9.

In [ ]:
# ============================================================
# Configuración base y entrenamiento del modelo base
# ============================================================
CONFIG_BASE = dict(hid_dim=256, capas=3, cabezas=8, pf_dim=512, dropout=0.1,
                   learning_rate=0.0005, batch_size=64, clip=1.0)
fila_base, historia_base, modelo_base = entrenar_y_registrar(CONFIG_BASE, verbose=True, forzar=True)
print(f"El modelo tiene {fila_base['parametros']:,} parámetros entrenables")

In [ ]:
# ============================================================
# Métricas del modelo base
# ============================================================
display(pd.DataFrame([fila_base]).round(4))

In [ ]:
# ============================================================
# Respuestas del modelo base para cinco entradas de validación
# ============================================================
ejemplos_val = MUESTRA_VAL[:5]
generadas_base = generar_respuestas(modelo_base, ejemplos_val)
display(pd.DataFrame({
    "entrada": [" ".join(s[:-1]) for s, _, _ in ejemplos_val],
    "respuesta_real": [" ".join(t[1:-1]) for _, _, t in ejemplos_val],
    "respuesta_generada": [" ".join(g) for g in generadas_base],
}))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Pérdida, perplejidad, exactitud por token, BLEU, ROUGE-L y distinct-2 del modelo base.
- Calidad de las cinco respuestas: ¿responden a la entrada o son genéricas?

## 8. Rango de error entre reentrenamientos

Se reentrena `CONFIG_BASE` con otras cuatro semillas (5 en total). El **rango de la pérdida de validación** se guarda en `MARGEN_ERROR`: diferencias menores entre variantes son empate técnico. Con menos semillas el rango se subestima, porque crece con el número de repeticiones; por eso se usan 5, igual que en la clasificación con RNN. También se reporta la variación del BLEU.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración base con distintas semillas
# ============================================================
SEMILLAS_ADICIONALES = [1, 2, 3, 4]
filas_variabilidad = [{"semilla": SEED, **fila_base}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad.append({"semilla": semilla, **fila})
tabla_variabilidad = pd.DataFrame(filas_variabilidad)

In [ ]:
# ============================================================
# Resultados por semilla
# ============================================================
display(tabla_variabilidad.round(4))

In [ ]:
# ============================================================
# Estadísticos de variación y rango de error
# ============================================================
estadisticos_variabilidad = tabla_variabilidad[["loss_val", "ppl_val", "bleu", "rouge_l", "distinct_2", "mejor_epoca"]].agg(
    ["mean", "std", "min", "max"]).T
estadisticos_variabilidad["rango"] = estadisticos_variabilidad["max"] - estadisticos_variabilidad["min"]
MARGEN_ERROR = estadisticos_variabilidad.loc["loss_val", "rango"]
media_loss_base = estadisticos_variabilidad.loc["loss_val", "mean"]
print(f"Rango de error de la pérdida de validación (MARGEN_ERROR): {MARGEN_ERROR:.4f}")
print(f"Rango del BLEU entre semillas: {estadisticos_variabilidad.loc['bleu', 'rango'] * 100:.2f} puntos")
display(estadisticos_variabilidad.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Rango de la pérdida y del BLEU entre semillas; cuán estable es el BLEU como criterio.

## 9. Visualización de la máscara y de la predicción del siguiente token

Con el modelo base y una frase del problema: entrada **«how are you ?»** y respuesta parcial **«<bos> i am»**.

### 9.1 La máscara del decoder

Cada fila es el token que consulta y cada columna el token que puede mirar: 1 = permitido, 0 = bloqueado (futuro).

In [ ]:
# ============================================================
# Máscara causal para «<bos> i am fine .»
# ============================================================
frase_trg = ["<bos>", "i", "am", "fine", "."]
idx = torch.tensor([[vocab[t] for t in frase_trg]], device=device)
mascara = modelo_base.make_trg_mask(idx)[0, 0].int().cpu().numpy()
display(pd.DataFrame(mascara, index=frase_trg, columns=frase_trg))

### 9.2 Predicción del siguiente token

Se entregan «how are you ?» al encoder y «<bos> i am» al decoder, y se muestran los cinco tokens más probables después de «am», con su logit y su probabilidad.

In [ ]:
# ============================================================
# Logits y probabilidades del siguiente token
# ============================================================
modelo_base.eval()
tokens_frase = ["how", "are", "you", "?", "<eos>"]
src_idx = torch.tensor([[vocab[t] for t in tokens_frase]], device=device)
trg_idx = torch.tensor([[vocab[t] for t in ["<bos>", "i", "am"]]], device=device)
with torch.no_grad():
    salida, _ = modelo_base(src_idx, trg_idx)
logits = salida[0, -1]
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 5)
itos = vocab.get_itos()
display(pd.DataFrame({"token": [itos[i] for i in top.indices.tolist()],
                      "logit": [round(logits[i].item(), 2) for i in top.indices.tolist()],
                      "probabilidad": [f"{p:.1%}" for p in top.values.tolist()]}))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Tokens más probables después de «i am» y si tienen sentido como respuesta a «how are you ?».

## 10. Experimentos de ajuste

Cada experimento cambia **un único hiperparámetro** respecto de `CONFIG_ACTUAL`; la variante elegida se incorpora a la configuración. Cada experimento se divide en tres celdas: entrenamiento, tabla y selección. El criterio es la **menor pérdida de validación**, con BLEU y ROUGE-L como confirmación y `MARGEN_ERROR` para detectar empates. Si la mejor variante queda en el borde del rango, `ampliar_rango` agrega el valor siguiente. Como la elección secuencial depende del orden, la sección 16 verifica las combinaciones sospechosas.

In [ ]:
# ============================================================
# Configuración actual (se actualiza al final de cada experimento)
# ============================================================
CONFIG_ACTUAL = dict(CONFIG_BASE)
mostrar_config(CONFIG_ACTUAL)

### 10.1 Cabezas de atención (`cabezas`)

Con `hid_dim` fijo, más cabezas significa más subespacios de atención, pero cada uno más pequeño (`head_dim` = 128, 64, 32 o 16). Se limita a 16 cabezas para que cada una tenga al menos 16 dimensiones.

In [ ]:
# ============================================================
# Experimento 1: cabezas
# ============================================================
variantes_cabezas = {
    "2": dict(cabezas=2),
    "4": dict(cabezas=4),
    "8": dict(cabezas=8),
}
tabla_cabezas = ejecutar_experimento("cabezas", variantes_cabezas, CONFIG_ACTUAL)
tabla_cabezas = ampliar_rango("cabezas", variantes_cabezas, CONFIG_ACTUAL, "cabezas",
                              arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=1, maximo=16)

In [ ]:
# ============================================================
# Resultados del experimento 1
# ============================================================
display(tabla_cabezas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 1
# ============================================================
elegida = seleccionar_variante(tabla_cabezas)
# elegida = "8"  # Cambio Manual
aplicar_variante("cabezas", variantes_cabezas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `cabezas` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

### 10.2 Número de bloques (`capas`)

Cantidad de bloques en el encoder y en el decoder. Más bloques dan más capacidad, a cambio de tiempo y riesgo de sobreajuste.

In [ ]:
# ============================================================
# Experimento 2: capas
# ============================================================
variantes_capas = {
    "1": dict(capas=1),
    "2": dict(capas=2),
    "3": dict(capas=3),
}
tabla_capas = ejecutar_experimento("capas", variantes_capas, CONFIG_ACTUAL)
tabla_capas = ampliar_rango("capas", variantes_capas, CONFIG_ACTUAL, "capas",
                            arriba=lambda v: v + 1, abajo=lambda v: v - 1, minimo=1, maximo=6)

In [ ]:
# ============================================================
# Resultados del experimento 2
# ============================================================
display(tabla_capas[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 2
# ============================================================
elegida = seleccionar_variante(tabla_capas)
# elegida = "3"  # Cambio Manual
aplicar_variante("capas", variantes_capas, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `capas` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

### 10.3 Dimensión del modelo (`hid_dim`)

Dimensión de los embeddings y de todos los vectores internos. **No se amplía a 512**: con 512 los parámetros y el tiempo por época se triplican aproximadamente, y con ≈ 76.000 pares de entrenamiento el riesgo de sobreajuste aumenta. Con 8 cabezas, `hid_dim = 128` deja cabezas de solo 16 dimensiones; esa combinación se revisa en la sección 16.

In [ ]:
# ============================================================
# Experimento 3: hid_dim
# ============================================================
variantes_hid_dim = {
    "128": dict(hid_dim=128),
    "256": dict(hid_dim=256),
}
tabla_hid_dim = ejecutar_experimento("hid_dim", variantes_hid_dim, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 3
# ============================================================
display(tabla_hid_dim[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 3
# ============================================================
elegida = seleccionar_variante(tabla_hid_dim)
# elegida = "256"  # Cambio Manual
aplicar_variante("hid_dim", variantes_hid_dim, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `hid_dim` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

### 10.4 Dimensión del feed-forward (`pf_dim`)

Tamaño de la capa interna del MLP de cada bloque.

In [ ]:
# ============================================================
# Experimento 4: pf_dim
# ============================================================
variantes_pf_dim = {
    "512": dict(pf_dim=512),
    "1024": dict(pf_dim=1024),
}
tabla_pf_dim = ejecutar_experimento("pf_dim", variantes_pf_dim, CONFIG_ACTUAL)
tabla_pf_dim = ampliar_rango("pf_dim", variantes_pf_dim, CONFIG_ACTUAL, "pf_dim",
                             arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=256, maximo=4096)

In [ ]:
# ============================================================
# Resultados del experimento 4
# ============================================================
display(tabla_pf_dim[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 4
# ============================================================
elegida = seleccionar_variante(tabla_pf_dim)
# elegida = "512"  # Cambio Manual
aplicar_variante("pf_dim", variantes_pf_dim, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `pf_dim` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

### 10.5 Dropout

Regularización en embeddings, atención y subcapas. 0,0 es el límite natural inferior.

In [ ]:
# ============================================================
# Experimento 5: dropout
# ============================================================
variantes_dropout = {
    "0.1": dict(dropout=0.1),
    "0.3": dict(dropout=0.3),
}
tabla_dropout = ejecutar_experimento("dropout", variantes_dropout, CONFIG_ACTUAL)
tabla_dropout = ampliar_rango("dropout", variantes_dropout, CONFIG_ACTUAL, "dropout",
                              arriba=lambda v: round(v + 0.1, 1), abajo=lambda v: round(v - 0.1, 1), minimo=0.0, maximo=0.5)

In [ ]:
# ============================================================
# Resultados del experimento 5
# ============================================================
display(tabla_dropout[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 5
# ============================================================
elegida = seleccionar_variante(tabla_dropout)
# elegida = "0.1"  # Cambio Manual
aplicar_variante("dropout", variantes_dropout, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `dropout` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

### 10.6 Tasa de aprendizaje

Los Transformers entrenados sin calentamiento de la tasa se vuelven inestables con tasas altas; se prueba en torno a 0,0005, incluido el valor por defecto de Adam (0,001).

In [ ]:
# ============================================================
# Experimento 6: learning_rate
# ============================================================
variantes_learning_rate = {
    "0.0001": dict(learning_rate=0.0001),
    "0.0005": dict(learning_rate=0.0005),
    "0.001": dict(learning_rate=0.001),
}
tabla_learning_rate = ejecutar_experimento("learning_rate", variantes_learning_rate, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 6
# ============================================================
display(tabla_learning_rate[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 6
# ============================================================
elegida = seleccionar_variante(tabla_learning_rate)
# elegida = "0.0005"  # Cambio Manual
aplicar_variante("learning_rate", variantes_learning_rate, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de la tasa en la convergencia, la pérdida y la generación.
- Si alguna variante no logra entrenar (pérdida que no baja, respuestas vacías o repetidas), se registra como **Error / Falla**.

### 10.7 Tamaño del lote

Con 76.052 pares, los lotes de 32, 64 y 128 producen ≈ 2.377, 1.189 y 595 actualizaciones por época.

In [ ]:
# ============================================================
# Experimento 7: batch_size
# ============================================================
variantes_batch_size = {
    "32": dict(batch_size=32),
    "64": dict(batch_size=64),
    "128": dict(batch_size=128),
}
tabla_batch_size = ejecutar_experimento("batch_size", variantes_batch_size, CONFIG_ACTUAL)
tabla_batch_size = ampliar_rango("batch_size", variantes_batch_size, CONFIG_ACTUAL, "batch_size",
                                 arriba=lambda v: v * 2, abajo=lambda v: v // 2, minimo=16, maximo=256)

In [ ]:
# ============================================================
# Resultados del experimento 7
# ============================================================
display(tabla_batch_size[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 7
# ============================================================
elegida = seleccionar_variante(tabla_batch_size)
# elegida = "64"  # Cambio Manual
aplicar_variante("batch_size", variantes_batch_size, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Efecto de `batch_size` en la pérdida, el BLEU, el ROUGE-L, la diversidad y el tiempo; si alguna diferencia supera `MARGEN_ERROR`; criterio usado si hubo empate.

## 11. Modelo final y estrategia de decodificación

### 11.1 Configuración final

La configuración final es `CONFIG_ACTUAL` después de todos los experimentos.

In [ ]:
# ============================================================
# Configuración final
# ============================================================
CONFIG_FINAL = dict(CONFIG_ACTUAL)
mostrar_config(CONFIG_FINAL)

In [ ]:
# ============================================================
# Decisiones tomadas
# ============================================================
display(pd.DataFrame({"experimento": list(decisiones), "variante_elegida": list(decisiones.values())}))

### 11.2 Entrenamiento del modelo final

Se entrena con **el mismo régimen que todos los experimentos** (máximo 15 épocas, paciencia 2, con extensión si llega al tope) y la semilla principal, para que su resultado sea comparable con el de la selección. Se entrena siempre, porque se necesitan sus pesos.

In [ ]:
# ============================================================
# Entrenamiento del modelo final
# ============================================================
fila_final, historia_final, modelo_final = entrenar_y_registrar(CONFIG_FINAL, verbose=True, forzar=True)
print(f"El modelo final tiene {fila_final['parametros']:,} parámetros entrenables")

In [ ]:
# ============================================================
# Métricas del modelo final
# ============================================================
display(pd.DataFrame([fila_final]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo final
# ============================================================
graficar_curvas(historia_final, "Transformer final")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Si el modelo final repite su corrida de la selección; mejor época y distancia entre las curvas.

### 11.3 Estrategia de decodificación

La decodificación no cambia el modelo, solo **cómo se elige** cada token al generar. Con el modelo final y la muestra de validación se comparan:

- **Greedy:** siempre el token más probable. Respuestas seguras, pero repetitivas.
- **Sorteo top-k = 10 con temperatura 0,7 y 1,0:** se sortea entre los 10 tokens más probables.

Se espera un compromiso entre **coincidencia con la referencia** (BLEU, ROUGE) y **diversidad** (distinct). La elección es caso a caso y queda en la línea `# Cambio Manual`.

In [ ]:
# ============================================================
# Comparación de estrategias de decodificación (sin reentrenar)
# ============================================================
ESTRATEGIAS = {
    "greedy": dict(temperatura=None, top_k=None),
    "top-k 10, temperatura 0.7": dict(temperatura=0.7, top_k=10),
    "top-k 10, temperatura 1.0": dict(temperatura=1.0, top_k=10),
}
filas_decodificacion = []
for etiqueta, parametros in ESTRATEGIAS.items():
    torch.manual_seed(SEED)
    generadas = generar_respuestas(modelo_final, MUESTRA_VAL, **parametros)
    filas_decodificacion.append({"estrategia": etiqueta, **evaluar_generacion(REFERENCIAS_MUESTRA, generadas)})
tabla_decodificacion = pd.DataFrame(filas_decodificacion)
display(tabla_decodificacion.round(4))

In [ ]:
# ============================================================
# Respuestas de cada estrategia para las mismas entradas
# ============================================================
torch.manual_seed(SEED)
display(pd.DataFrame({
    "entrada": [" ".join(s[:-1]) for s, _, _ in ejemplos_val],
    **{etiqueta: [" ".join(g) for g in generar_respuestas(modelo_final, ejemplos_val, **parametros)]
       for etiqueta, parametros in ESTRATEGIAS.items()},
}))

In [ ]:
# ============================================================
# Estrategia elegida
# ============================================================
ESTRATEGIA_FINAL = "greedy"  # Cambio Manual
print("Estrategia de decodificación elegida:", ESTRATEGIA_FINAL)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Compromiso entre BLEU/ROUGE y distinct-2 según la estrategia, y criterio de la elección.

## 12. Evaluación en test

El conjunto de test se usa una sola vez, con el modelo final y la estrategia de decodificación elegida.

### 12.1 Métricas globales

Se generan las respuestas de los 6.740 pares de test. Como distinct depende del número de respuestas, se reporta también sobre una **muestra fija de 1.000 pares de test**, comparable con la muestra de validación.

In [ ]:
# ============================================================
# Métricas de generación en test
# ============================================================
criterio = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
test_loss, test_acc = evaluar(modelo_final, crear_loader(test_list, 128), criterio)
torch.manual_seed(SEED)
generadas_test = generar_respuestas(modelo_final, test_list, **ESTRATEGIAS[ESTRATEGIA_FINAL])
referencias_test = [trg[1:-1] for _, _, trg in test_list]
INDICES_MUESTRA_TEST = random.Random(SEED).sample(range(len(test_list)), 1000)


def metricas_test(nombre, generadas):
    completo = {"modelo": nombre, "conjunto": "test completo (6.740)", **evaluar_generacion(referencias_test, generadas)}
    muestra = {"modelo": nombre, "conjunto": "test, muestra de 1.000",
               **evaluar_generacion([referencias_test[i] for i in INDICES_MUESTRA_TEST],
                                    [generadas[i] for i in INDICES_MUESTRA_TEST])}
    return [completo, muestra]


filas_test_transformer = metricas_test("Transformer final", generadas_test)
tabla_test_transformer = pd.DataFrame(filas_test_transformer)
tabla_test_transformer["loss_test"] = test_loss
tabla_test_transformer["ppl_test"] = math.exp(test_loss)
tabla_test_transformer["exactitud_token_test"] = test_acc
print(f"Largo medio de las respuestas reales: {np.mean([len(r) for r in referencias_test]):.1f} tokens")
display(tabla_test_transformer.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- BLEU, ROUGE y diversidad en test; distinct de la muestra de 1.000 frente al de validación; largo medio generado frente al real.

### 12.2 Ejemplos representativos

Se calcula el BLEU de cada respuesta (`sentence_bleu`) y su ROUGE-L. Se muestran las 10 mejores, 10 al azar sin coincidencia (ROUGE-L = 0) y las respuestas que más repite el modelo. Los errores se clasifican en: respuesta **genérica**, **incoherente**, **repetitiva** o **cortada**.

In [ ]:
# ============================================================
# Tabla de respuestas de test con su BLEU y ROUGE-L
# ============================================================
ejemplos_test = pd.DataFrame({
    "dialogo_id": pares_test["dialogo_id"].values,
    "entrada": [" ".join(s[:-1]) for s, _, _ in test_list],
    "respuesta_real": [" ".join(r) for r in referencias_test],
    "respuesta_generada": [" ".join(g) for g in generadas_test],
    "bleu_oracion": [sentence_bleu([r], g, smoothing_function=SUAVIZADO) for r, g in zip(referencias_test, generadas_test)],
    "rouge_l": [rouge_l(r, g) for r, g in zip(referencias_test, generadas_test)],
    "acto_entrada": pares_test["acto_entrada"].values,
})
print(f"Respuestas con ROUGE-L = 0: {(ejemplos_test['rouge_l'] == 0).mean():.1%}")

In [ ]:
# ============================================================
# 10 mejores respuestas según ROUGE-L
# ============================================================
display(ejemplos_test.sort_values(["rouge_l", "bleu_oracion"], ascending=False).head(10).round(3))

In [ ]:
# ============================================================
# 10 respuestas al azar sin coincidencia (ROUGE-L = 0)
# ============================================================
display(ejemplos_test[ejemplos_test["rouge_l"] == 0].sample(10, random_state=SEED).round(3))

In [ ]:
# ============================================================
# Respuestas que más repite el modelo
# ============================================================
frecuentes_generadas = ejemplos_test["respuesta_generada"].value_counts().head(10).to_frame("veces")
frecuentes_generadas["% de las respuestas"] = (frecuentes_generadas["veces"] / len(ejemplos_test) * 100).round(2)
display(frecuentes_generadas)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Aciertos: en qué tipo de entradas coincide con la referencia.
- Errores: proporción de respuestas genéricas, incoherentes, repetitivas o cortadas, y si algunas respuestas sin coincidencia son en realidad válidas.

### 12.3 Desempeño según el acto del turno de entrada

Métricas por separado según el acto del turno de entrada. BLEU y ROUGE usan todos los pares del grupo; **distinct se calcula sobre una muestra del mismo tamaño en cada grupo** (el del grupo más pequeño), para que sea comparable.

In [ ]:
# ============================================================
# Métricas por acto del turno de entrada
# ============================================================
grupos = {acto: np.where(ejemplos_test["acto_entrada"].values == acto)[0] for acto in ACTOS}
n_comun = min(len(v) for v in grupos.values())
filas_acto = []
for acto, indices in grupos.items():
    refs = [referencias_test[i] for i in indices]
    hips = [generadas_test[i] for i in indices]
    submuestra = random.Random(SEED).sample(list(indices), n_comun)
    fila = {"acto_entrada": acto, "pares": len(indices), **evaluar_generacion(refs, hips)}
    fila["distinct_1"] = distinct_n([generadas_test[i] for i in submuestra], 1)
    fila["distinct_2"] = distinct_n([generadas_test[i] for i in submuestra], 2)
    filas_acto.append(fila)
tabla_por_acto = pd.DataFrame(filas_acto)
print(f"distinct calculado sobre {n_comun} respuestas por acto")
display(tabla_por_acto.round(4))

In [ ]:
# ============================================================
# Gráfico de BLEU-1 y ROUGE-L por acto
# ============================================================
por_acto = tabla_por_acto.melt(id_vars="acto_entrada", value_vars=["bleu_1", "rouge_l"], var_name="metrica", value_name="valor")
fig = px.bar(por_acto, x="acto_entrada", y="valor", color="metrica", barmode="group", color_discrete_sequence=COLORES[1::-1],
             title="BLEU-1 y ROUGE-L según el acto del turno de entrada (test)",
             labels={"acto_entrada": "acto del turno de entrada", "metrica": "métrica"})
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Tipo de turno donde el modelo responde mejor y dónde peor, y si la diversidad cambia según el acto.

### 12.4 Mapas de atención cruzada

Para dos entradas de test con forma de pregunta se muestra la atención decoder → encoder de la última capa: filas = tokens generados, columnas = tokens de entrada. Una celda oscura indica que, al generar ese token, el modelo se fijó en esa palabra de la entrada.

In [ ]:
# ============================================================
# Selección de ejemplos para la atención
# ============================================================
EJEMPLOS_ATENCION = [i for i, (s, _, _) in enumerate(test_list)
                     if pares_test.loc[i, "dialogo_id"] >= 10 and "?" in s and 6 <= len(s) <= 12][:2]
for i in EJEMPLOS_ATENCION:
    print(i, "|", " ".join(test_list[i][0]))

In [ ]:
# ============================================================
# Atención cruzada: ejemplo 1
# ============================================================
src, _, trg = test_list[EJEMPLOS_ATENCION[0]]
respuesta, atencion = generar_con_atencion(modelo_final, src)
print("entrada  :", " ".join(src))
print("real     :", " ".join(trg[1:-1]))
print("generada :", " ".join(respuesta))
mostrar_atencion(src, respuesta, atencion, "Atención cruzada decoder → encoder (última capa)").show()

In [ ]:
# ============================================================
# Atención cruzada: ejemplo 2
# ============================================================
src, _, trg = test_list[EJEMPLOS_ATENCION[1]]
respuesta, atencion = generar_con_atencion(modelo_final, src)
print("entrada  :", " ".join(src))
print("real     :", " ".join(trg[1:-1]))
print("generada :", " ".join(respuesta))
mostrar_atencion(src, respuesta, atencion, "Atención cruzada decoder → encoder (última capa)").show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Si alguna cabeza se concentra en palabras clave de la entrada (por ejemplo, la palabra interrogativa) y si la atención se reparte al generar `<eos>`.

## 13. Comparación con un Seq2Seq LSTM

Para comparar arquitecturas con **evidencia propia**, se entrena un encoder-decoder recurrente con **los mismos pares, el mismo vocabulario, la misma muestra de evaluación, el mismo régimen de épocas y las mismas funciones** de entrenamiento y generación.

| Aspecto | Seq2Seq LSTM | Transformer |
|---|---|---|
| Procesamiento | Secuencial: un token después de otro | Paralelo: todos los tokens a la vez (en entrenamiento) |
| Contexto de la entrada | Un **vector de contexto fijo** (estado final del encoder) | **Un vector por token**, consultado con atención en cada paso |
| Dependencias largas | Se debilitan con la distancia | Cualquier par de tokens está a un paso de atención |
| Orden de las palabras | Implícito en la recurrencia | Explícito con embeddings de posición |
| Costo | Lineal en el largo, pero no paralelizable | Cuadrático en el largo, pero paralelizable en GPU |

### 13.1 Modelo Seq2Seq LSTM

- **Encoder:** `Embedding` + `LSTM` de 2 capas; con `pack_padded_sequence` el estado final corresponde al último token real. Su estado final (h, c) es el vector de contexto.
- **Decoder:** `Embedding` + `LSTM` de 2 capas que parte del estado del encoder, y `Linear` sobre el vocabulario.
- Recibe `(src, trg)` y devuelve `(logits, None)`, por lo que reutiliza `entrenar_y_registrar` y `generar_lote`.
- Configuración: embedding de 256 y estado oculto de 512 (capacidad comparable a la del Transformer) y tasa de aprendizaje 0,001, el valor por defecto de Adam, estable para redes recurrentes. **Esta configuración no se ajustó** con experimentos; es una limitación de la comparación.

In [ ]:
# ============================================================
# Seq2Seq LSTM (encoder-decoder recurrente sin atención)
# ============================================================
class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim, hid_dim, capas, dropout, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.emb_encoder = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_decoder = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.encoder = nn.LSTM(embed_dim, hid_dim, num_layers=capas, batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(embed_dim, hid_dim, num_layers=capas, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(hid_dim, vocab_size)

    def forward(self, src, trg):
        largos = (src != self.pad_idx).sum(1).cpu()
        empaquetado = pack_padded_sequence(self.dropout(self.emb_encoder(src)), largos, batch_first=True, enforce_sorted=False)
        _, estado = self.encoder(empaquetado)
        # estado = (h, c), cada uno [capas, batch size, hid dim]: el vector de contexto
        salida, _ = self.decoder(self.dropout(self.emb_decoder(trg)), estado)
        return self.fc_out(salida), None


def construir_lstm(config):
    return LSTMSeq2Seq(len(vocab), config["embed_dim"], config["hid_dim"], config["capas"], config["dropout"], PAD_IDX).to(device)


CONFIG_LSTM = dict(modelo="lstm", embed_dim=256, hid_dim=512, capas=2, dropout=0.1, learning_rate=0.001, batch_size=64, clip=1.0)
mostrar_config(CONFIG_LSTM)

### 13.2 Entrenamiento del Seq2Seq LSTM

Se entrena con el mismo régimen que el Transformer (15 épocas, paciencia 2, con extensión). Se entrena siempre, porque se necesitan sus pesos.

In [ ]:
# ============================================================
# Entrenamiento del Seq2Seq LSTM
# ============================================================
fila_lstm, historia_lstm, modelo_lstm = entrenar_y_registrar(CONFIG_LSTM, constructor=construir_lstm, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Métricas de validación del Seq2Seq LSTM
# ============================================================
display(pd.DataFrame([fila_lstm]).round(4))

### 13.3 Evaluación del Seq2Seq LSTM en test

Misma estrategia de decodificación y mismas muestras que el Transformer.

In [ ]:
# ============================================================
# Métricas de test del Seq2Seq LSTM
# ============================================================
test_loss_lstm, test_acc_lstm = evaluar(modelo_lstm, crear_loader(test_list, 128), criterio)
torch.manual_seed(SEED)
generadas_lstm = generar_respuestas(modelo_lstm, test_list, **ESTRATEGIAS[ESTRATEGIA_FINAL])
tabla_test_lstm = pd.DataFrame(metricas_test("Seq2Seq LSTM", generadas_lstm))
tabla_test_lstm["loss_test"] = test_loss_lstm
tabla_test_lstm["ppl_test"] = math.exp(test_loss_lstm)
tabla_test_lstm["exactitud_token_test"] = test_acc_lstm
display(tabla_test_lstm.round(4))

### 13.4 Comparación

Métricas de test (muestra de 1.000 para la diversidad), costo de entrenamiento y respuestas de ambos modelos para las mismas entradas.

In [ ]:
# ============================================================
# Tabla comparativa Transformer vs. Seq2Seq LSTM
# ============================================================
comparacion = pd.concat([tabla_test_transformer, tabla_test_lstm], ignore_index=True)
comparacion = comparacion[comparacion["conjunto"] == "test completo (6.740)"].set_index("modelo")
muestra = pd.concat([tabla_test_transformer, tabla_test_lstm])
muestra = muestra[muestra["conjunto"] == "test, muestra de 1.000"].set_index("modelo")
comparacion[["distinct_1", "distinct_2"]] = muestra[["distinct_1", "distinct_2"]]
for nombre, fila in [("Transformer final", fila_final), ("Seq2Seq LSTM", fila_lstm)]:
    for clave in ["parametros", "segundos_por_epoca", "mejor_epoca", "segundos"]:
        comparacion.loc[nombre, clave] = fila[clave]
comparacion = comparacion.drop(columns="conjunto")
display(comparacion.round(4))

In [ ]:
# ============================================================
# Gráfico de métricas: Transformer vs. Seq2Seq LSTM
# ============================================================
metricas_grafico = ["bleu", "bleu_1", "rouge_1", "rouge_2", "rouge_l", "distinct_1", "distinct_2"]
datos_comparacion = comparacion[metricas_grafico].reset_index().melt(id_vars="modelo", var_name="metrica", value_name="valor")
fig = px.bar(datos_comparacion, x="metrica", y="valor", color="modelo", barmode="group",
             color_discrete_sequence=[COLORES[0], COLORES[2]], title="Transformer vs. Seq2Seq LSTM en test",
             labels={"metrica": "métrica"})
fig.show()

In [ ]:
# ============================================================
# Respuestas de ambos modelos para 8 entradas de test al azar
# ============================================================
indices = ejemplos_test.sample(8, random_state=SEED).index
display(pd.DataFrame({
    "entrada": ejemplos_test.loc[indices, "entrada"],
    "respuesta_real": ejemplos_test.loc[indices, "respuesta_real"],
    "transformer": ejemplos_test.loc[indices, "respuesta_generada"],
    "lstm": [" ".join(generadas_lstm[i]) for i in indices],
}))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencias en pérdida, BLEU, ROUGE y diversidad, y si superan el rango de error.
- Costo: parámetros, segundos por época y épocas hasta converger.
- Ventajas, limitaciones y contextos de uso de cada arquitectura según la evidencia.

## 14. Comparación entre la configuración base y la final

Qué cambió desde `CONFIG_BASE` y cuánto aportó cada decisión. Una decisión **supera el margen** si la diferencia de pérdida entre la mejor y la peor variante de su experimento es mayor que `MARGEN_ERROR`. La comparación de esta sección usa una sola ejecución del modelo final; la comparación con 5 semillas está en la sección 16.3.

In [ ]:
# ============================================================
# Configuración base vs. final
# ============================================================
config_base_vs_final = pd.DataFrame({"base": pd.Series(CONFIG_BASE).astype(str), "final": pd.Series(CONFIG_FINAL).astype(str)})
config_base_vs_final["cambio"] = config_base_vs_final["base"] != config_base_vs_final["final"]
display(config_base_vs_final)

In [ ]:
# ============================================================
# Aporte de cada decisión
# ============================================================
filas_decision = []
for tabla in resultados_experimentos:
    nombre = tabla.loc[0, "experimento"]
    if nombre in decisiones:
        rango = tabla["loss_val"].max() - tabla["loss_val"].min()
        loss_elegida = tabla.loc[tabla["variante"] == decisiones[nombre], "loss_val"].iloc[0]
        filas_decision.append({"experimento": nombre, "variante_elegida": decisiones[nombre], "loss_elegida": loss_elegida,
                               "mejor_variante": tabla.loc[0, "variante"], "rango_entre_variantes": rango,
                               "supera_margen": rango > MARGEN_ERROR})
tabla_decisiones = pd.DataFrame(filas_decision)
print(f"Pérdida de validación: base {media_loss_base:.4f} (media de 5 semillas) → final {fila_final['loss_val']:.4f} "
      f"(una ejecución; {fila_final['loss_val'] - media_loss_base:+.4f})")
display(tabla_decisiones.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Decisiones que superaron el rango de error y qué parte de la mejora explica cada una.

## 15. Síntesis de los experimentos

Cada punto es una variante; el eje horizontal es su distancia en pérdida de validación a la mejor variante de su experimento. Las variantes a la izquierda de la línea amarilla están en **empate técnico** con la mejor. La tabla completa queda en el respaldo.

In [ ]:
# ============================================================
# Síntesis gráfica de todos los experimentos
# ============================================================
resumen_general = pd.concat(resultados_experimentos, ignore_index=True)
TOPE_GRAFICO = 0.15
resumen_general["eleccion"] = ["elegida" if decisiones.get(e) == v else "otra"
                                for e, v in zip(resumen_general["experimento"], resumen_general["variante"])]
resumen_general["diferencia_grafico"] = resumen_general["diferencia_vs_mejor"].clip(upper=TOPE_GRAFICO)
resumen_general["etiqueta"] = [f"{v} (+{d:.2f})" if d > TOPE_GRAFICO else v
                               for v, d in zip(resumen_general["variante"], resumen_general["diferencia_vs_mejor"])]
fig = px.scatter(resumen_general, x="diferencia_grafico", y="experimento", color="eleccion", symbol="eleccion",
                 text="etiqueta", color_discrete_map={"elegida": AMARILLO, "otra": COLORES[0]},
                 hover_data=["diferencia_vs_mejor", "loss_val", "bleu", "rouge_l", "parametros", "segundos_por_epoca", "origen"],
                 title="Distancia de cada variante a la mejor de su experimento (pérdida de validación)",
                 labels={"diferencia_grafico": "diferencia con la mejor variante", "eleccion": "variante"})
fig.add_vline(x=MARGEN_ERROR, line_dash="dash", line_color=AMARILLO, annotation_text="rango de error")
fig.update_traces(textposition="top center")
fig.update_layout(height=60 * resumen_general["experimento"].nunique() + 150,
                  yaxis={"categoryorder": "array", "categoryarray": list(dict.fromkeys(resumen_general["experimento"]))[::-1]})
fig.show()

Las variantes que superan una diferencia de 0,15 (por ejemplo, una tasa de aprendizaje que no logra entrenar) se dibujan en el borde derecho, con su diferencia real entre paréntesis, para no comprimir el resto del gráfico.

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Cuántas decisiones quedaron en empate técnico y en cuáles la variante elegida no fue la de menor pérdida.

## 16. Verificaciones de la dependencia del orden

La selección de un hiperparámetro a la vez es *greedy*: cada decisión se tomó con los valores elegidos hasta ese momento. Los experimentos anteriores no se rehacen; aquí se prueban, sin usar test, las combinaciones donde la interacción es más probable, y el modelo final con las mismas 5 semillas que el base:

| Verificación | Sospecha |
|---|---|
| 16.1 Menos bloques con la configuración final | Las capas se eligieron antes de ampliar el feed-forward; un feed-forward más ancho podría compensar un bloque menos |
| 16.2 `hid_dim = 128` con 4 cabezas | Con 8 cabezas, `hid_dim = 128` deja cabezas de 16 dimensiones; con 4 vuelven a 32, como en la base |
| 16.3 Configuración final con 5 semillas | Comparación justa con la media de 5 semillas del base |

In [ ]:
# ============================================================
# Verificaciones 16.1 y 16.2
# ============================================================
VERIFICACIONES = {
    "16.1 Un bloque menos con la configuración final": dict(capas=max(1, CONFIG_FINAL["capas"] - 1)),
    "16.2 hid_dim 128 con 4 cabezas": dict(hid_dim=128, cabezas=4),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final}]
for nombre, cambios in VERIFICACIONES.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion = pd.DataFrame(filas_verificacion)
tabla_verificacion["diferencia_vs_final"] = tabla_verificacion["loss_val"] - fila_final["loss_val"]
tabla_verificacion["supera_margen"] = tabla_verificacion["diferencia_vs_final"].abs() > MARGEN_ERROR

In [ ]:
# ============================================================
# Resultados de las verificaciones 16.1 y 16.2
# ============================================================
display(tabla_verificacion[["verificacion", "loss_val", "ppl_val", "bleu", "rouge_l", "distinct_2", "mejor_epoca",
                            "parametros", "segundos_por_epoca", "diferencia_vs_final", "supera_margen", "origen"]].round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Si alguna combinación mejora a la configuración final por más del rango de error (el orden sí influyó) o queda en empate.

### 16.3 Configuración final con 5 semillas

Se reentrena `CONFIG_FINAL` con las mismas semillas adicionales que el base para comparar medias en igualdad de condiciones.

In [ ]:
# ============================================================
# Configuración final con las mismas semillas que el base
# ============================================================
filas_final_semillas = [{"semilla": SEED, **fila_final}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas.append({"semilla": semilla, **fila})
tabla_final_semillas = pd.DataFrame(filas_final_semillas)

In [ ]:
# ============================================================
# Comparación justa: base vs. final (media de 5 semillas)
# ============================================================
columnas = ["loss_val", "ppl_val", "bleu", "rouge_l", "distinct_2"]
comparacion_justa = pd.DataFrame({
    "base": tabla_variabilidad[columnas].agg(["mean", "std", "min", "max"]).stack(),
    "final": tabla_final_semillas[columnas].agg(["mean", "std", "min", "max"]).stack(),
}).swaplevel().sort_index()
comparacion_justa["diferencia"] = comparacion_justa["final"] - comparacion_justa["base"]
display(comparacion_justa.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mejora real de la configuración final frente al base con 5 semillas en ambos, comparada con la de una sola ejecución (sección 14).

## 17. Conclusiones

*(completar después de ejecutar)*

- **Componentes que más influyeron:** …
- **Calidad de las respuestas:** …
- **Transformer frente a Seq2Seq LSTM:** …
- **Dependencia del orden:** …
- **Limitaciones:** …

### 17.1 Mejoras propuestas

- Usar **dos o tres turnos previos** como entrada, en vez de solo el anterior, para dar al modelo el contexto de la conversación.
- **Beam search** en la decodificación, para buscar respuestas más probables en conjunto y no token a token.
- Tasa de aprendizaje con **calentamiento** (*warmup*) y **suavizado de etiquetas**, como en el paper original, que podrían estabilizar tasas más altas.
- Ajustar un modelo **preentrenado** (por ejemplo, T5 o un modelo conversacional) y comparar con este.
- Eliminar los pares repetidos entre particiones (69 diálogos de validación y 94 de test son idénticos a uno de `train`).

## 18. Conclusión final

*(completar después de ejecutar)*

## 19. Referencias

- Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
- Li, Y. et al. (2017). *DailyDialog: A Manually Labelled Multi-turn Dialogue Dataset*. IJCNLP.
- Papineni, K. et al. (2002). *BLEU: a Method for Automatic Evaluation of Machine Translation*. ACL.
- Lin, C.-Y. (2004). *ROUGE: A Package for Automatic Evaluation of Summaries*. ACL Workshop.
- Li, J. et al. (2016). *A Diversity-Promoting Objective Function for Neural Conversation Models*. NAACL (distinct-n).
- Cho, A. et al. (2024). *Transformer Explainer: Interactive Learning of Text-Generative Models*. arXiv:2408.04619.
- Kingma, D. P. y Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR.
- *Implementación de un Transformer con PyTorch* (`GP_3_5_4`), basado en material del Instituto Humai. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).
- *Transformer paso a paso*, *Transformer primera parte* y *Transformer segunda parte*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).
- *Encode-Decode Seq2Seq* y *Traductor con mecanismos de atención*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).

## 20. Anexo A — Síntesis visual para la presentación

Gráficos de los resultados principales, con formato 16:9 para la presentación. Se generan desde las variables ya calculadas, sin entrenar nada. Cada gráfico se guarda en `respaldo_ep3_transformer/graficos/` como **HTML interactivo**; el botón de cámara del gráfico descarga un **PNG de 1600 × 900**. Si la librería `kaleido` está disponible, el PNG también se guarda directamente.

In [ ]:
# ============================================================
# Estilo y exportación de los gráficos de la presentación
# ============================================================
RUTA_RESPALDO = "respaldo_ep3_transformer"
RUTA_GRAFICOS = f"{RUTA_RESPALDO}/graficos"
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
AZUL_MARINO = "#0E2747"
ANCHO_PNG, ALTO_PNG = 1600, 900


def estilo_presentacion(fig, titulo, subtitulo=None):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:15px;color:#555555'>{subtitulo}</span>"
    fig.update_layout(template="plotly_white", title={"text": texto, "x": 0.02},
                      font={"family": "Calibri, Arial", "size": 16, "color": AZUL_MARINO},
                      height=560, margin={"t": 110, "l": 80, "r": 40, "b": 80},
                      legend={"orientation": "h", "y": -0.18, "title": None})
    return fig


def exportar(fig, nombre):
    fig.write_html(f"{RUTA_GRAFICOS}/{nombre}.html", include_plotlyjs="cdn",
                   config={"toImageButtonOptions": {"format": "png", "width": ANCHO_PNG, "height": ALTO_PNG,
                                                    "scale": 2, "filename": nombre}})
    try:
        fig.write_image(f"{RUTA_GRAFICOS}/{nombre}.png", width=ANCHO_PNG, height=ALTO_PNG, scale=2)
        print(f"{nombre}: guardado en HTML y PNG")
    except Exception:
        print(f"{nombre}: guardado en HTML (PNG desde el botón de cámara del gráfico)")

### A.1 Modelo base frente a modelo final (5 semillas)

Pérdida de validación de cada semilla, con la media y el rango de error del base. Mensaje: la mejora del modelo final frente al ruido.

In [ ]:
# ============================================================
# A.1 Base frente a final con 5 semillas
# ============================================================
base = tabla_variabilidad["loss_val"].to_numpy()
final = tabla_final_semillas["loss_val"].to_numpy()
fig = go.Figure()
fig.add_hrect(y0=base.min(), y1=base.max(), fillcolor=AMARILLO, opacity=0.25, line_width=0,
              annotation_text=f"rango de error del base ({MARGEN_ERROR:.4f})", annotation_position="top left")
for etiqueta, valores, color in [("Base", base, COLORES[2]), ("Final", final, AZUL_MARINO)]:
    fig.add_trace(go.Scatter(x=[etiqueta] * len(valores), y=valores, mode="markers", name=etiqueta,
                             marker={"size": 14, "color": color, "line": {"width": 1, "color": "white"}}))
    fig.add_trace(go.Scatter(x=[etiqueta], y=[valores.mean()], mode="markers+text", showlegend=False,
                             marker={"size": 40, "symbol": "line-ew-open", "color": AZUL_MARINO, "line": {"width": 4}},
                             text=[f"media {valores.mean():.4f}"], textposition="middle right"))
fig.update_yaxes(title_text="pérdida de validación (menor es mejor)")
estilo_presentacion(fig, f"Pérdida de validación: {final.mean() - base.mean():+.4f} frente al base",
                    "Cinco semillas por configuración, mismas semillas en ambas")
exportar(fig, "01_base_vs_final_semillas")
fig.show()

### A.2 Impacto de cada decisión

Diferencia de pérdida entre la mejor y la peor variante de cada experimento, en escala logarítmica, frente al rango de error.

In [ ]:
# ============================================================
# A.2 Impacto de cada decisión frente al rango de error
# ============================================================
datos = tabla_decisiones.assign(resultado=np.where(tabla_decisiones["supera_margen"], "supera el rango de error", "empate técnico"))
fig = px.bar(datos, x="experimento", y="rango_entre_variantes", color="resultado", log_y=True,
             text=datos["rango_entre_variantes"].round(4),
             color_discrete_map={"supera el rango de error": AZUL_MARINO, "empate técnico": COLORES[2]},
             labels={"rango_entre_variantes": "mejor − peor variante (pérdida, escala log)", "experimento": ""})
fig.add_hline(y=MARGEN_ERROR, line_dash="dash", line_color=AMARILLO, annotation_text=f"rango de error: {MARGEN_ERROR:.4f}")
estilo_presentacion(fig, "Qué decisiones superan el ruido", "Diferencia de pérdida de validación entre la mejor y la peor variante")
exportar(fig, "02_impacto_decisiones")
fig.show()

### A.3 Pérdida frente a BLEU

Cada punto es una variante de los experimentos (sin las que no lograron entrenar). Mensaje: una menor pérdida no garantiza un mejor BLEU.

In [ ]:
# ============================================================
# A.3 Pérdida de validación frente a BLEU
# ============================================================
datos = resumen_general[resumen_general["loss_val"] < 4]
fig = px.scatter(datos, x="loss_val", y="bleu", color="experimento", text="variante",
                 hover_data=["rouge_l", "distinct_2"],
                 labels={"loss_val": "pérdida de validación (menor es mejor)", "bleu": "BLEU (muestra de validación)"})
fig.update_traces(marker={"size": 14}, textposition="top center")
estilo_presentacion(fig, "Menor pérdida no implica mejor BLEU", "Todas las variantes de los experimentos de ajuste")
exportar(fig, "03_perdida_vs_bleu")
fig.show()

### A.4 Curvas del modelo final

Pérdida y perplejidad por época del Transformer final.

In [ ]:
# ============================================================
# A.4 Curvas del modelo final
# ============================================================
epocas = list(range(1, len(historia_final["loss_val"]) + 1))
mejor = int(np.argmin(historia_final["loss_val"])) + 1
fig = make_subplots(rows=1, cols=2, subplot_titles=["Pérdida", "Perplejidad"])
for col, transformar in [(1, np.array), (2, np.exp)]:
    fig.add_trace(go.Scatter(x=epocas, y=transformar(historia_final["loss_train"]), name="entrenamiento",
                             line_color=COLORES[2], legendgroup="train", showlegend=col == 1), row=1, col=col)
    fig.add_trace(go.Scatter(x=epocas, y=transformar(historia_final["loss_val"]), name="validación",
                             line_color=AZUL_MARINO, legendgroup="val", showlegend=col == 1), row=1, col=col)
    fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
    fig.update_xaxes(title_text="época", row=1, col=col)
estilo_presentacion(fig, f"Transformer final: mejor época {mejor}", "Pérdida y perplejidad de entrenamiento y validación")
exportar(fig, "04_curvas_final")
fig.show()

### A.5 Estrategia de decodificación

Coincidencia con la referencia (BLEU-1, ROUGE-L) frente a diversidad (distinct-2) para cada estrategia.

In [ ]:
# ============================================================
# A.5 Estrategias de decodificación
# ============================================================
datos = tabla_decodificacion.melt(id_vars="estrategia", value_vars=["bleu_1", "rouge_l", "distinct_2"],
                                  var_name="metrica", value_name="valor")
fig = px.bar(datos, x="estrategia", y="valor", color="metrica", barmode="group", text=datos["valor"].round(3),
             color_discrete_sequence=[COLORES[2], COLORES[1], AZUL_MARINO], labels={"valor": "", "estrategia": ""})
estilo_presentacion(fig, "Decodificación: coincidencia frente a diversidad", "Muestra fija de 1.000 pares de validación")
exportar(fig, "05_decodificacion")
fig.show()

### A.6 Desempeño según el acto del turno de entrada

BLEU-1 y ROUGE-L en test según el acto del turno de entrada.

In [ ]:
# ============================================================
# A.6 Métricas por acto del turno de entrada
# ============================================================
datos = tabla_por_acto.melt(id_vars="acto_entrada", value_vars=["bleu_1", "rouge_l"], var_name="metrica", value_name="valor")
fig = px.bar(datos, x="acto_entrada", y="valor", color="metrica", barmode="group", text=datos["valor"].round(3),
             color_discrete_sequence=[COLORES[1], AZUL_MARINO], category_orders={"acto_entrada": ACTOS},
             labels={"valor": "", "acto_entrada": "acto del turno de entrada"})
estilo_presentacion(fig, "Qué turnos responde mejor el modelo", "Test; BLEU-1 y ROUGE-L por acto de diálogo de la entrada")
exportar(fig, "06_por_acto")
fig.show()

### A.7 Transformer frente a Seq2Seq LSTM: métricas

Métricas de test de ambos modelos (diversidad sobre la muestra de 1.000).

In [ ]:
# ============================================================
# A.7 Transformer frente a Seq2Seq LSTM: métricas de test
# ============================================================
datos = comparacion[["bleu", "bleu_1", "rouge_l", "distinct_2"]].reset_index().melt(id_vars="modelo", var_name="metrica", value_name="valor")
fig = px.bar(datos, x="metrica", y="valor", color="modelo", barmode="group", text=datos["valor"].round(3),
             color_discrete_map={"Transformer final": AZUL_MARINO, "Seq2Seq LSTM": COLORES[2]},
             labels={"valor": "", "metrica": ""})
estilo_presentacion(fig, "Transformer frente a Seq2Seq LSTM", "Test; mismos datos, vocabulario, régimen de entrenamiento y decodificación")
exportar(fig, "07_transformer_vs_lstm_metricas")
fig.show()

### A.8 Transformer frente a Seq2Seq LSTM: costo y pérdida

Pérdida de test, parámetros y tiempo por época de ambos modelos.

In [ ]:
# ============================================================
# A.8 Transformer frente a Seq2Seq LSTM: costo y pérdida
# ============================================================
modelos = list(comparacion.index)
colores = [AZUL_MARINO if m == "Transformer final" else COLORES[2] for m in modelos]
fig = make_subplots(rows=1, cols=3, subplot_titles=["Pérdida de test", "Parámetros (millones)", "Segundos por época"])
for col, (columna, factor, formato) in enumerate([("loss_test", 1, ".3f"), ("parametros", 1e-6, ".1f"),
                                                  ("segundos_por_epoca", 1, ".0f")], start=1):
    valores = comparacion[columna].astype(float) * factor
    fig.add_trace(go.Bar(x=modelos, y=valores, marker_color=colores, showlegend=False,
                         text=[format(v, formato) for v in valores]), row=1, col=col)
estilo_presentacion(fig, "Costo y pérdida de cada arquitectura", "Menor es mejor en los tres paneles")
exportar(fig, "08_transformer_vs_lstm_costo")
fig.show()

### A.9 Mapa de atención

Atención cruzada de la última capa para el primer ejemplo de la sección 12.4.

In [ ]:
# ============================================================
# A.9 Mapa de atención cruzada
# ============================================================
src, _, _ = test_list[EJEMPLOS_ATENCION[0]]
respuesta, atencion = generar_con_atencion(modelo_final, src)
fig = mostrar_atencion(src, respuesta, atencion, "")
estilo_presentacion(fig, "Dónde mira el decoder al generar cada palabra", f"Entrada: «{' '.join(src[:-1])}»")
fig.update_layout(height=700)
exportar(fig, "09_mapa_atencion")
fig.show()

## 21. Anexo B — Herramientas y tecnologías de apoyo utilizadas

En el desarrollo de este trabajo se utilizaron las siguientes herramientas y tecnologías:

**Entornos de ejecución:**

- Google Colab (GPU NVIDIA T4) — entorno de ejecución del notebook.
- Visual Studio Code con la extensión Google Colab — edición y ejecución del notebook conectado a Colab.

**Asistente de IA:**

- Claude (Opus 5.5), Anthropic — apoyo en la estructuración del notebook, el código y la redacción.

**Herramienta de estudio:**

- Transformer Explainer — [https://poloclub.github.io/transformer-explainer/](https://poloclub.github.io/transformer-explainer/)

**Repositorio:**

- GitHub — [https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY010_003D](https://github.com/EstebanPachecoVega/EV3_EPACHECO_FFIGUEROA_DLY010_003D)

**Librerías y frameworks (sitios oficiales):**

- PyTorch — [https://pytorch.org/](https://pytorch.org/)
- NLTK — [https://www.nltk.org/](https://www.nltk.org/)
- NumPy — [https://numpy.org/](https://numpy.org/)
- Pandas — [https://pandas.pydata.org/](https://pandas.pydata.org/)
- Plotly (Python) — [https://plotly.com/python/](https://plotly.com/python/)

## 22. Respaldo de resultados

Se guardan las tablas, las respuestas generadas, los pesos de ambos modelos, el JSON de corridas actualizado y los gráficos del anexo A en `respaldo_ep3_transformer/`, y se comprime en `respaldo_ep3_transformer.zip`. En Colab web, el `.zip` se descarga con `from google.colab import files; files.download("respaldo_ep3_transformer.zip")`.

In [ ]:
# ============================================================
# Respaldo de resultados
# ============================================================
os.makedirs(RUTA_RESPALDO, exist_ok=True)
resumen_general.to_csv(f"{RUTA_RESPALDO}/experimentos_todas_las_ejecuciones.csv", index=False)
tabla_variabilidad.to_csv(f"{RUTA_RESPALDO}/variabilidad.csv", index=False)
estadisticos_variabilidad.to_csv(f"{RUTA_RESPALDO}/estadisticos_variabilidad.csv")
config_base_vs_final.to_csv(f"{RUTA_RESPALDO}/config_base_vs_final.csv")
tabla_decisiones.to_csv(f"{RUTA_RESPALDO}/decisiones.csv", index=False)
tabla_decodificacion.to_csv(f"{RUTA_RESPALDO}/decodificacion.csv", index=False)
tabla_verificacion.to_csv(f"{RUTA_RESPALDO}/verificaciones_orden.csv", index=False)
tabla_final_semillas.to_csv(f"{RUTA_RESPALDO}/final_5_semillas.csv", index=False)
comparacion_justa.to_csv(f"{RUTA_RESPALDO}/comparacion_base_final.csv")
pd.concat([tabla_test_transformer, tabla_test_lstm]).to_csv(f"{RUTA_RESPALDO}/metricas_test.csv", index=False)
tabla_por_acto.to_csv(f"{RUTA_RESPALDO}/metricas_por_acto.csv", index=False)
ejemplos_test.assign(respuesta_lstm=[" ".join(g) for g in generadas_lstm]).to_csv(
    f"{RUTA_RESPALDO}/respuestas_generadas_test.csv", index=False)
comparacion.to_csv(f"{RUTA_RESPALDO}/transformer_vs_lstm.csv")
with open(f"{RUTA_RESPALDO}/historias.json", "w") as f:
    json.dump({"base": historia_base, "transformer_final": historia_final, "lstm": historia_lstm}, f)
torch.save({"config": CONFIG_FINAL, "estado": modelo_final.state_dict(), "itos": vocab.get_itos()},
           f"{RUTA_RESPALDO}/transformer_final.pt")
torch.save({"config": CONFIG_LSTM, "estado": modelo_lstm.state_dict(), "itos": vocab.get_itos()},
           f"{RUTA_RESPALDO}/seq2seq_lstm.pt")
guardar_corridas()
shutil.copy(ARCHIVO_RESULTADOS, f"{RUTA_RESPALDO}/{ARCHIVO_RESULTADOS}")
shutil.make_archive(RUTA_RESPALDO, "zip", RUTA_RESPALDO)
print("Respaldo guardado en", RUTA_RESPALDO + ".zip")